# Fine tune PhoBERT-base-v2

In [ ]:
import torch
import transformers
import sklearn

print(f"PyTorch version   : {torch.__version__}")
print(f"Transformers ver  : {transformers.__version__}")
print(f"CUDA available    : {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU               : {torch.cuda.get_device_name(0)}")
    # Tổng VRAM (đơn vị GB)
    total_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"VRAM              : {total_mem:.1f} GB")
else:
    print("CẢNH BÁO: Không tìm thấy GPU, sẽ chạy trên CPU rất chậm!")

In [ ]:
# ── Standard library ──────────────────────────────────────
import os
import json
import warnings
warnings.filterwarnings("ignore")

# ── Data ─────────────────────────────────────────────────
import numpy as np
import pandas as pd

# ── ML metrics ───────────────────────────────────────────
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    f1_score,
)
from sklearn.utils.class_weight import compute_class_weight

# ── PyTorch ───────────────────────────────────────────────
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler

# ── HuggingFace Transformers ──────────────────────────────
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    get_linear_schedule_with_warmup,
)

# ── Reproducibility: fix random seed để kết quả lặp lại ──
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

print("Import xong.")

In [ ]:
# ── Đường dẫn ─────────────────────────────────────────────
# Sửa đường dẫn này cho phù hợp với Kaggle Dataset của bạn
DATA_PATH  = "/kaggle/input/datasets/tranquanghuy2809/tqhtqh/qwen_intent_classification.csv"
MODEL_PATH = "/kaggle/input/datasets/tranquanghuy2809/tqhtqh/phobert-base-v2"   # thư mục chứa model đã tải về
OUTPUT_DIR = "/kaggle/working/intent_model"     # nơi lưu model tốt nhất

# ── Label mapping ─────────────────────────────────────────
LABEL2ID = {"tra_cuu": 0, "tinh_toan": 1}
ID2LABEL  = {0: "tra_cuu", 1: "tinh_toan"}

# ── Hyperparameters ───────────────────────────────────────
MAX_LEN      = 128    # độ dài tối đa của câu sau tokenize
BATCH_SIZE   = 32     # số mẫu mỗi batch
NUM_EPOCHS   = 15     # số epoch train
LR           = 2e-5   # learning rate
WARMUP_RATIO = 0.1    # tỉ lệ warmup steps / total steps
WEIGHT_DECAY = 0.01   # L2 regularization, tránh overfitting

# ── Split ratio ───────────────────────────────────────────
VAL_RATIO  = 0.15
TEST_RATIO = 0.15
# → Train: 70%, Val: 15%, Test: 15%

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Sẽ train trên: {DEVICE}")

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Output dir: {OUTPUT_DIR}")

In [ ]:
df = pd.read_csv(DATA_PATH)

print(f"Tổng số mẫu: {len(df)}")
print(f"Các cột: {df.columns.tolist()}")
print()

# Đếm số lượng mỗi class
counts = df["intent"].value_counts()
print("Phân bố class:")
for intent, count in counts.items():
    pct = count / len(df) * 100
    print(f"  {intent:12s}: {count:4d} mẫu ({pct:.1f}%)")

ratio = counts["tra_cuu"] / counts["tinh_toan"]
print(f"\nTỉ lệ imbalance: {ratio:.1f}:1")

# Xem vài dòng mẫu
print("\nMột vài mẫu:")
print(df[["question_index", "question", "intent"]].head(8).to_string())

In [ ]:
# Gán group id cho mỗi câu
# question_index bắt đầu từ 1, nên trừ 1 trước khi chia
df["group"] = (df["question_index"] - 1) // 5

# Lấy danh sách unique groups
all_groups = df["group"].unique()
np.random.shuffle(all_groups)   # xáo trộn ngẫu nhiên

n_groups   = len(all_groups)
n_val      = int(n_groups * VAL_RATIO)
n_test     = int(n_groups * TEST_RATIO)
n_train    = n_groups - n_val - n_test

# Cắt danh sách groups thành 3 phần
train_groups = set(all_groups[:n_train])
val_groups   = set(all_groups[n_train:n_train + n_val])
test_groups  = set(all_groups[n_train + n_val:])

# Lọc dataframe theo group
df_train = df[df["group"].isin(train_groups)].reset_index(drop=True)
df_val   = df[df["group"].isin(val_groups)].reset_index(drop=True)
df_test  = df[df["group"].isin(test_groups)].reset_index(drop=True)

print(f"Tổng groups: {n_groups}")
print(f"  Train: {len(train_groups)} groups → {len(df_train)} mẫu")
print(f"  Val  : {len(val_groups)} groups → {len(df_val)} mẫu")
print(f"  Test : {len(test_groups)} groups → {len(df_test)} mẫu")
print()

# Kiểm tra phân bố class trong từng partition
for name, subset in [("Train", df_train), ("Val", df_val), ("Test", df_test)]:
    tt = (subset["intent"] == "tinh_toan").sum()
    tc = (subset["intent"] == "tra_cuu").sum()
    print(f"{name:5s} — tra_cuu: {tc:3d} | tinh_toan: {tt:2d} | ratio: {tc/max(tt,1):.1f}:1")

In [ ]:
class IntentDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_len):
        """
        dataframe: DataFrame có cột 'question' và 'intent'
        tokenizer: PhoBERT tokenizer
        max_len  : độ dài tối đa sau tokenize
        """
        self.texts     = dataframe["question"].tolist()
        self.labels    = [LABEL2ID[l] for l in dataframe["intent"].tolist()]
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        # DataLoader cần biết dataset có bao nhiêu mẫu
        return len(self.texts)

    def __getitem__(self, idx):
        # Được gọi mỗi khi DataLoader cần lấy 1 mẫu
        text = str(self.texts[idx])

        encoding = self.tokenizer(
            text,
            max_length=self.max_len,
            padding="max_length",   # độn đến max_len
            truncation=True,         # cắt nếu dài hơn max_len
            return_tensors="pt",     # trả về PyTorch tensor
        )

        return {
            # squeeze(0) để bỏ dimension batch thừa (shape: [1, max_len] → [max_len])
            "input_ids"     : encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "label"         : torch.tensor(self.labels[idx], dtype=torch.long),
        }

print("Định nghĩa IntentDataset xong.")

In [ ]:
print("Loading tokenizer...")
# Load từ local path vì Kaggle không có internet
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
print("Tokenizer loaded.")

# Tạo Dataset objects
train_dataset = IntentDataset(df_train, tokenizer, MAX_LEN)
val_dataset   = IntentDataset(df_val,   tokenizer, MAX_LEN)
test_dataset  = IntentDataset(df_test,  tokenizer, MAX_LEN)

# ── WeightedRandomSampler cho train set ──────────────────
# Tính trọng số: class ít mẫu hơn → trọng số cao hơn
train_labels  = np.array([LABEL2ID[l] for l in df_train["intent"]])
class_counts  = np.bincount(train_labels)          # [n_tra_cuu, n_tinh_toan]
class_weights = 1.0 / class_counts                 # nghịch đảo tần suất

# Mỗi mẫu trong train được gán trọng số theo class của nó
sample_weights = class_weights[train_labels]

sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True,   # cho phép lấy lại cùng mẫu (cần thiết để oversample)
)

# ── Tạo DataLoader ───────────────────────────────────────
# Train: dùng sampler, không dùng shuffle (sampler đã xử lý)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=sampler)
# Val/Test: shuffle=False để kết quả nhất quán
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches  : {len(val_loader)}")
print(f"Test batches : {len(test_loader)}")

# Kiểm tra một batch để chắc chắn shape đúng
sample_batch = next(iter(train_loader))
print(f"\nShape input_ids     : {sample_batch['input_ids'].shape}")
print(f"Shape attention_mask: {sample_batch['attention_mask'].shape}")
print(f"Shape labels        : {sample_batch['label'].shape}")

In [ ]:
print("Loading model...")

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_PATH,
    num_labels=2,
    id2label=ID2LABEL,
    label2id=LABEL2ID,
)
model = model.to(DEVICE)
print("Model loaded.")

# Đếm tổng số tham số
total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Tổng tham số     : {total_params:,}")
print(f"Tham số trainable: {trainable_params:,}")

# ── Loss function với class weight ───────────────────────
# compute_class_weight trả về weight cho mỗi class
# "balanced" = n_samples / (n_classes * n_samples_per_class)
cw = compute_class_weight(
    class_weight="balanced",
    classes=np.array([0, 1]),
    y=train_labels,
)
print(f"\nClass weights (từ compute_class_weight):")
print(f"  tra_cuu  (0): {cw[0]:.4f}")
print(f"  tinh_toan(1): {cw[1]:.4f}")

criterion = nn.CrossEntropyLoss(
    weight=torch.tensor(cw, dtype=torch.float).to(DEVICE)
)

# ── Optimizer & Scheduler ────────────────────────────────
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LR,
    weight_decay=WEIGHT_DECAY,
)

total_steps   = len(train_loader) * NUM_EPOCHS
warmup_steps  = int(WARMUP_RATIO * total_steps)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps,
)

print(f"\nTotal training steps: {total_steps}")
print(f"Warmup steps        : {warmup_steps}")

In [ ]:
def evaluate(model, data_loader, device):
    """
    Chạy model trên toàn bộ data_loader, trả về các metric.
    """
    model.eval()  # chuyển sang eval mode: tắt dropout

    all_preds  = []
    all_labels = []

    with torch.no_grad():   # không tính gradient → tiết kiệm memory
        for batch in data_loader:
            input_ids      = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels         = batch["label"]         # giữ trên CPU để so sánh

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            # outputs.logits shape: [batch_size, num_labels]
            # argmax lấy class có logit cao nhất
            preds = outputs.logits.argmax(dim=-1).cpu().numpy()

            all_preds.extend(preds)
            all_labels.extend(labels.numpy())

    all_preds  = np.array(all_preds)
    all_labels = np.array(all_labels)

    accuracy    = (all_preds == all_labels).mean()
    f1_macro    = f1_score(all_labels, all_preds, average="macro")
    # average="binary" với pos_label=1: tính F1 riêng cho class tinh_toan
    f1_minority = f1_score(all_labels, all_preds, average="binary", pos_label=1)

    return {
        "accuracy"    : float(accuracy),
        "f1_macro"    : float(f1_macro),
        "f1_tinh_toan": float(f1_minority),
        "preds"       : all_preds,
        "labels"      : all_labels,
    }

print("Định nghĩa hàm evaluate xong.")

In [ ]:
best_f1    = 0.0
best_epoch = 0
history    = []  # lưu metric từng epoch để vẽ đồ thị sau

print(f"Bắt đầu train {NUM_EPOCHS} epochs...")
print(f"{'Epoch':>6} {'Loss':>8} {'Acc':>7} {'F1_mac':>8} {'F1_tinhToan':>12}")
print("-" * 50)

for epoch in range(1, NUM_EPOCHS + 1):

    # ── Train phase ───────────────────────────────────────
    model.train()
    total_loss = 0.0

    for batch in train_loader:
        input_ids      = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        labels         = batch["label"].to(DEVICE)

        # Xóa gradient từ bước trước, tránh tích lũy
        optimizer.zero_grad()

        # Forward: model trả về SequenceClassifierOutput, lấy .logits
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        loss    = criterion(outputs.logits, labels)

        # Backward: tính gradient
        loss.backward()

        # Clip gradient: nếu norm > 1.0 thì scale xuống
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        # Cập nhật weights và learning rate
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)

    # ── Val phase ─────────────────────────────────────────
    metrics = evaluate(model, val_loader, DEVICE)

    print(f"{epoch:>6} {avg_loss:>8.4f} {metrics['accuracy']:>7.4f} "
          f"{metrics['f1_macro']:>8.4f} {metrics['f1_tinh_toan']:>12.4f}")

    # Lưu lịch sử để plot sau
    history.append({
        "epoch"        : epoch,
        "loss"         : round(avg_loss, 4),
        "accuracy"     : round(metrics["accuracy"], 4),
        "f1_macro"     : round(metrics["f1_macro"], 4),
        "f1_tinh_toan" : round(metrics["f1_tinh_toan"], 4),
    })

    # Lưu model tốt nhất theo F1 macro
    if metrics["f1_macro"] > best_f1:
        best_f1    = metrics["f1_macro"]
        best_epoch = epoch
        model.save_pretrained(OUTPUT_DIR)
        tokenizer.save_pretrained(OUTPUT_DIR)

print("-" * 50)
print(f"Best: epoch {best_epoch}, F1 macro = {best_f1:.4f}")
print(f"Model đã lưu tại: {OUTPUT_DIR}")

In [ ]:
import matplotlib.pyplot as plt

epochs_list    = [h["epoch"]         for h in history]
losses         = [h["loss"]          for h in history]
f1_macros      = [h["f1_macro"]      for h in history]
f1_tinh_toans  = [h["f1_tinh_toan"]  for h in history]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Đồ thị 1: Training Loss
ax1.plot(epochs_list, losses, "b-o", markersize=4)
ax1.set_title("Training Loss")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")
ax1.grid(True, alpha=0.3)

# Đồ thị 2: F1 scores trên Val set
ax2.plot(epochs_list, f1_macros,     "g-o", markersize=4, label="F1 Macro")
ax2.plot(epochs_list, f1_tinh_toans, "r-s", markersize=4, label="F1 tinh_toan")
ax2.axvline(x=best_epoch, color="orange", linestyle="--", label=f"Best epoch ({best_epoch})")
ax2.set_title("F1 Score trên Val Set")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("F1")
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "training_curve.png"), dpi=150)
plt.show()
print("Đã lưu training_curve.png")

In [ ]:
print("Loading best model để đánh giá trên test set...")
best_model = AutoModelForSequenceClassification.from_pretrained(OUTPUT_DIR).to(DEVICE)

test_metrics = evaluate(best_model, test_loader, DEVICE)

print("\n" + "=" * 50)
print("KẾT QUẢ TRÊN TEST SET (best model)")
print("=" * 50)
print(f"  Accuracy    : {test_metrics['accuracy']:.4f}")
print(f"  F1 Macro    : {test_metrics['f1_macro']:.4f}")
print(f"  F1 tinh_toan: {test_metrics['f1_tinh_toan']:.4f}")

# Classification report chi tiết: precision, recall, F1 từng class
print("\nClassification Report:")
print(classification_report(
    test_metrics["labels"],
    test_metrics["preds"],
    target_names=["tra_cuu", "tinh_toan"]
))

# Confusion matrix: hàng = ground truth, cột = predicted
# [TN  FP]
# [FN  TP]
cm = confusion_matrix(test_metrics["labels"], test_metrics["preds"])
print("Confusion Matrix:")
print(f"              Pred tra_cuu  Pred tinh_toan")
print(f"True tra_cuu  {cm[0][0]:>10}  {cm[0][1]:>14}")
print(f"True tinh_toan{cm[1][0]:>10}  {cm[1][1]:>14}")

In [ ]:
TEST_CSV_PATH = "/kaggle/input/datasets/tranquanghuy2809/tqhtqh/question_classifier.csv"

In [ ]:
LABEL2ID = {"tra_cuu": 0, "tinh_toan": 1}
ID2LABEL  = {0: "tra_cuu", 1: "tinh_toan"}
MAX_LEN   = 128

# ── Load CSV ─────────────────────────────────────────────────────────────────
df_ext = pd.read_csv(TEST_CSV_PATH)
print(f"File test: {len(df_ext)} câu")
print(df_ext["intent"].value_counts().to_string())
print()

# ── Tạo dataset & dataloader ─────────────────────────────────────────────────
# Dùng lại class IntentDataset đã định nghĩa ở cell trước
ext_dataset = IntentDataset(df_ext, tokenizer, MAX_LEN)
ext_loader  = torch.utils.data.DataLoader(ext_dataset, batch_size=32, shuffle=False)

# ── Chạy evaluate ────────────────────────────────────────────────────────────
best_model.eval()
all_preds  = []
all_labels = []

with torch.no_grad():
    for batch in ext_loader:
        input_ids      = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        outputs = best_model(input_ids=input_ids, attention_mask=attention_mask)
        preds   = outputs.logits.argmax(dim=-1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(batch["label"].numpy())

all_preds  = np.array(all_preds)
all_labels = np.array(all_labels)

# ── In kết quả ───────────────────────────────────────────────────────────────
accuracy    = (all_preds == all_labels).mean()
f1_macro    = f1_score(all_labels, all_preds, average="macro")
f1_minority = f1_score(all_labels, all_preds, average="binary", pos_label=1)

print("=" * 50)
print("KẾT QUẢ TRÊN FILE CSV TEST RIÊNG")
print("=" * 50)
print(f"  Accuracy    : {accuracy:.4f}")
print(f"  F1 Macro    : {f1_macro:.4f}")
print(f"  F1 tinh_toan: {f1_minority:.4f}")

print("\nClassification Report:")
print(classification_report(all_labels, all_preds, target_names=["tra_cuu", "tinh_toan"]))

cm = confusion_matrix(all_labels, all_preds)
print("Confusion Matrix:")
print(f"              Pred tra_cuu  Pred tinh_toan")
print(f"True tra_cuu  {cm[0][0]:>10}  {cm[0][1]:>14}")
print(f"True tinh_toan{cm[1][0]:>10}  {cm[1][1]:>14}")

# ── Xem những câu bị predict sai ─────────────────────────────────────────────
wrong_idx = np.where(all_preds != all_labels)[0]
print(f"\nSố câu sai: {len(wrong_idx)}/{len(all_labels)}")
if len(wrong_idx) > 0:
    print("\nCác câu predict sai:")
    for i in wrong_idx:
        true_label = ID2LABEL[all_labels[i]]
        pred_label = ID2LABEL[all_preds[i]]
        question   = df_ext.iloc[i]["question"]
        short_q    = question[:80] + "..." if len(question) > 80 else question
        print(f"  [{i}] True: {true_label:10s} | Pred: {pred_label:10s} | {short_q}")


In [ ]:
import numpy as np
from sklearn.metrics import f1_score

# Chạy model trên val set, lấy xác suất thay vì argmax
best_model.eval()
all_probs  = []
all_labels = []

with torch.no_grad():
    for batch in val_loader:
        input_ids      = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        outputs = best_model(input_ids=input_ids, attention_mask=attention_mask)
        # softmax → xác suất, lấy cột 1 (tinh_toan)
        probs = torch.softmax(outputs.logits, dim=-1)[:, 1].cpu().numpy()
        all_probs.extend(probs)
        all_labels.extend(batch["label"].numpy())

all_probs  = np.array(all_probs)
all_labels = np.array(all_labels)

# Thử các threshold từ 0.3 đến 0.9
print(f"{'Threshold':>10} {'F1_macro':>10} {'F1_tinh_toan':>13} {'Precision':>10} {'Recall':>8}")
print("-" * 55)
for threshold in np.arange(0.3, 0.91, 0.05):
    preds = (all_probs >= threshold).astype(int)
    f1_mac = f1_score(all_labels, preds, average="macro")
    f1_min = f1_score(all_labels, preds, average="binary", pos_label=1)
    from sklearn.metrics import precision_score, recall_score
    prec = precision_score(all_labels, preds, pos_label=1, zero_division=0)
    rec  = recall_score(all_labels, preds, pos_label=1, zero_division=0)
    print(f"{threshold:>10.2f} {f1_mac:>10.4f} {f1_min:>13.4f} {prec:>10.4f} {rec:>8.4f}")

# Contrastive Foundation

## Imports

In [1]:
import os, json, time, random
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup,
)

print(f"PyTorch: {torch.__version__}")
print(f"CUDA:    {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU:     {torch.cuda.get_device_name(0)}")
    print(f"VRAM:    {torch.cuda.get_device_properties(0).total_memory/1024**3:.1f}GB")

PyTorch: 2.10.0+cu128
CUDA:    True
GPU:     NVIDIA RTX PRO 6000 Blackwell Server Edition
VRAM:    95.0GB


## Path & Configs

In [2]:
# Model paths 
MINILM_BASE  = "/kaggle/input/datasets/tranquanghuy2809/tqhtqh/ms-marco-MiniLM-L12-v2"
BGE_DIR      = "/kaggle/input/datasets/tranquanghuy2809/tqhtqh/bge-reranker-v2-m3"
PHORANKER    = "/kaggle/input/datasets/tranquanghuy2809/tqhtqh/PhoRanker"
STAGE_A_CKPT = "/kaggle/input/datasets/tranquanghuy2809/tqhtqh/stage_a_no_mmarco"

# Data paths 
DOMAIN_TRAIN = "/kaggle/input/datasets/tranquanghuy2809/tqhtqh/domain_train_final_train.jsonl"
DOMAIN_DEV   = "/kaggle/input/datasets/tranquanghuy2809/tqhtqh/domain_train_final_dev.jsonl"
MMARCO       = "/kaggle/input/datasets/tranquanghuy2809/tqhtqh/reranker_data/train_triplets.jsonl"
RERANK_991   = "/kaggle/input/datasets/tranquanghuy2809/tqhtqh/retrieve_rerank_991.jsonl"
GOLD_CHUNKS  = "/kaggle/input/datasets/tranquanghuy2809/tqhtqh/gold_chunks_judged.jsonl"
MARGIN_TRAIN = "/kaggle/input/datasets/tranquanghuy2809/tqhtqh/domain_train_with_teacher_scores.jsonl" 

# Test paths 
TEST_Q       = "/kaggle/input/datasets/tranquanghuy2809/tqhtqh/question.json"
RETRIEVE_TEST = "/kaggle/input/datasets/tranquanghuy2809/tqhtqh/retrieve_test.jsonl"

# Output dirs 
CKPT_STAGE_A_WITH_MMARCO    = "/kaggle/working/ablation/stage_a_with_mmarco"
CKPT_STAGE_A_NO_MMARCO      = "/kaggle/working/ablation/stage_a_no_mmarco"
CKPT_STAGE_B_LISTWISE       = "/kaggle/working/ablation/stage_b_listwise"
CKPT_STAGE_B_MARGIN_MSE     = "/kaggle/working/ablation/stage_b_margin_mse"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
print("\nPaths configured ✓")

Device: cuda

Paths configured ✓


## Classes (Datasets, Loss, Eval)

In [3]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False


# PairwiseDataset 
class PairwiseDataset(Dataset):
    def __init__(self, paths, tokenizer, max_length=512):
        self.data = []
        for p in paths:
            with open(p) as f:
                for line in f:
                    self.data.append(json.loads(line))
        self.tok     = tokenizer
        self.max_len = max_length

    def encode(self, query, passage):
        return self.tok(
            query, passage,
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )

    def __getitem__(self, idx):
        d   = self.data[idx]
        pos = self.encode(d["query"], d["positive"])
        neg = self.encode(d["query"], d["negative"])
        return {
            "pos_input_ids":      pos["input_ids"].squeeze(),
            "pos_attention_mask": pos["attention_mask"].squeeze(),
            "neg_input_ids":      neg["input_ids"].squeeze(),
            "neg_attention_mask": neg["attention_mask"].squeeze(),
        }

    def __len__(self):
        return len(self.data)


class ListwiseKDDataset(Dataset):
    def __init__(self, rerank_path, tokenizer,
                 max_length=512, max_candidates=20):
        self.tok      = tokenizer
        self.max_len  = max_length
        self.max_cand = max_candidates
        self.records  = []
        with open(rerank_path) as f:
            for line in f:
                self.records.append(json.loads(line))
        print(f"KD records: {len(self.records)}")

    def __getitem__(self, idx):
        d          = self.records[idx]
        query      = d["question"]
        candidates = d["candidates"][:self.max_cand]
        encodings, scores = [], []
        for c in candidates:
            enc = self.tok(
                query, c["chunk"],
                max_length=self.max_len,
                padding="max_length",
                truncation=True,
                return_tensors="pt",
            )
            encodings.append({
                "input_ids":      enc["input_ids"].squeeze(),
                "attention_mask": enc["attention_mask"].squeeze(),
            })
            scores.append(c["bge_score"])
        return {
            "encodings":  encodings,
            "bge_scores": torch.tensor(scores, dtype=torch.float),
        }

    def __len__(self):
        return len(self.records)

# RankNet/ADR-MSE 
class ListwiseRankDataset(Dataset):
    def __init__(self, rerank_path, tokenizer,    
                 max_length=512, max_candidates=20):
        self.tok      = tokenizer
        self.max_len  = max_length
        self.max_cand = max_candidates
        self.records  = []
        with open(rerank_path) as f:
            for line in f:
                self.records.append(json.loads(line))
        print(f"Total records: {len(self.records)}")

    def __getitem__(self, idx):
        d          = self.records[idx]
        query      = d["question"]
        candidates = d["candidates"][:self.max_cand]

        encodings, bge_scores, ranks = [], [], []
        for c in candidates:
            enc = self.tok(
                query, c["chunk"],
                max_length=self.max_len,
                padding="max_length",
                truncation=True,
                return_tensors="pt",
            )
            encodings.append({
                "input_ids":      enc["input_ids"].squeeze(),
                "attention_mask": enc["attention_mask"].squeeze(),
            })
            bge_scores.append(c["bge_score"])
            ranks.append(c["rank"])  # 0=best, 19=worst

        return {
            "encodings":  encodings,
            "bge_scores": torch.tensor(bge_scores, dtype=torch.float),
            "ranks":      torch.tensor(ranks,      dtype=torch.float),
        }

    def __len__(self):
        return len(self.records)

def collate_listwise_rank(batch):
    all_ids, all_masks, all_scores, all_ranks, sizes = [], [], [], [], []
    for item in batch:
        n = len(item["encodings"])
        sizes.append(n)
        for enc in item["encodings"]:
            all_ids.append(enc["input_ids"])
            all_masks.append(enc["attention_mask"])
        all_scores.append(item["bge_scores"])
        all_ranks.append(item["ranks"])
    return {
        "input_ids":      torch.stack(all_ids),
        "attention_mask": torch.stack(all_masks),
        "bge_scores":     all_scores,   # list of tensors
        "ranks":          all_ranks,    # list of tensors
        "sizes":          sizes,
    }


# ADR-MSE Loss 
class ADRMSELoss(nn.Module):
    """Asymmetric Distillation Ranking MSE
    Student margin matches teacher RANK margin (normalized)
    rank: 0=best → normalize về [-1, +1]
    """
    def __init__(self):
        super().__init__()
        self.mse = nn.MSELoss()

    def forward(self, student_logits, teacher_ranks):
        # teacher_ranks: [0, 1, 2, ..., 19] → normalize về [-1, +1]
        # rank 0 (best) → +1, rank 19 (worst) → -1
        n = teacher_ranks.size(-1)
        rank_signal = 1.0 - 2.0 * teacher_ranks / (n - 1)  # [+1, ..., -1]

        # Normalize student logits về cùng scale
        s_mean = student_logits.mean(dim=-1, keepdim=True)
        s_std  = student_logits.std(dim=-1, keepdim=True) + 1e-8
        student_norm = (student_logits - s_mean) / s_std

        return self.mse(student_norm, rank_signal)


# RankNet Loss với teacher soft labels 
class RankNetSoftLoss(nn.Module):
    """RankNet (Burges et al., 2005) với teacher soft labels từ BGE-M3
    Thay vì binary label, dùng P_ij = σ(T_i - T_j) làm target
    """
    def __init__(self):
        super().__init__()

    def forward(self, student_logits, teacher_scores):
        n = student_logits.size(-1)

        # Tính teacher prob P_ij = σ(T_i - T_j) cho mọi cặp (i, j)
        # student_logits: [batch, n]
        s_diff = student_logits.unsqueeze(2) - student_logits.unsqueeze(1)  # [B, n, n]
        t_diff = teacher_scores.unsqueeze(2) - teacher_scores.unsqueeze(1)  # [B, n, n]

        teacher_prob = torch.sigmoid(t_diff)  # P_ij từ teacher
        student_prob = torch.sigmoid(s_diff)  # Predicted P_ij từ student

        # BCE loss trên tất cả các cặp, loại diagonal (i==j)
        mask = ~torch.eye(n, dtype=torch.bool, device=student_logits.device)
        loss = F.binary_cross_entropy(
            student_prob[:, mask],
            teacher_prob[:, mask],
        )
        return loss

def collate_listwise(batch):
    all_ids, all_masks, all_scores, sizes = [], [], [], []
    for item in batch:
        sizes.append(len(item["encodings"]))
        for enc in item["encodings"]:
            all_ids.append(enc["input_ids"])
            all_masks.append(enc["attention_mask"])
        all_scores.append(item["bge_scores"])
    return {
        "input_ids":      torch.stack(all_ids),
        "attention_mask": torch.stack(all_masks),
        "bge_scores":     all_scores,
        "sizes":          sizes,
    }


# MarginMSEDataset (Stage B Margin-MSE) 
class MarginMSEDataset(Dataset):
    def __init__(self, path, tokenizer, max_length=512):
        self.data = []
        with open(path) as f:
            for line in f:
                self.data.append(json.loads(line))
        self.tok     = tokenizer
        self.max_len = max_length

    def encode(self, query, passage):
        return self.tok(
            query, passage,
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )

    def __getitem__(self, idx):
        d   = self.data[idx]
        pos = self.encode(d["query"], d["positive"])
        neg = self.encode(d["query"], d["negative"])
        return {
            "pos_input_ids":      pos["input_ids"].squeeze(),
            "pos_attention_mask": pos["attention_mask"].squeeze(),
            "neg_input_ids":      neg["input_ids"].squeeze(),
            "neg_attention_mask": neg["attention_mask"].squeeze(),
            "teacher_margin":     torch.tensor(d["teacher_margin"], dtype=torch.float),
        }

    def __len__(self):
        return len(self.data)


# Loss functions
class StageALoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.bce = nn.BCEWithLogitsLoss()

    def forward(self, pos_logits, neg_logits):
        return (self.bce(pos_logits, torch.ones_like(pos_logits)) +
                self.bce(neg_logits, torch.zeros_like(neg_logits))) / 2


class ListwiseKLLoss(nn.Module):
    def __init__(self, temperature=2.0):
        super().__init__()
        self.T = temperature

    def forward(self, student_logits, teacher_scores):
        s = F.log_softmax(student_logits / self.T, dim=-1)
        t = F.softmax(teacher_scores    / self.T, dim=-1)
        return F.kl_div(s, t, reduction="batchmean") * (self.T ** 2)


class MarginMSELoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.mse = nn.MSELoss()

    def forward(self, pos_logits, neg_logits, teacher_margin):
        return self.mse(pos_logits - neg_logits, teacher_margin)


# Dev evaluation (pairwise accuracy)
def evaluate_pairwise(model, dev_loader, device):
    model.eval()
    correct = total = 0
    with torch.no_grad():
        for batch in dev_loader:
            pos = model(
                input_ids=batch["pos_input_ids"].to(device),
                attention_mask=batch["pos_attention_mask"].to(device),
            ).logits
            neg = model(
                input_ids=batch["neg_input_ids"].to(device),
                attention_mask=batch["neg_attention_mask"].to(device),
            ).logits
            correct += (pos > neg).sum().item()
            total   += pos.size(0)
    return correct / total


print("Cell 03 done — classes & losses loaded ✓")

Cell 03 done — classes & losses loaded ✓


## Train stage A

In [ ]:
def train_stage_a(
    domain_train_path,
    mmarco_path,          # None = không dùng mMARCO
    dev_path,
    output_dir,
    base_model=None,
    epochs=5,
    batch_size=32,
    lr=2e-5,
    max_length=512,
    domain_upsample=8,
    patience=2,
    seed=42,
):
    if base_model is None:
        base_model = MINILM_BASE

    set_seed(seed)
    tokenizer = AutoTokenizer.from_pretrained(base_model)
    model     = AutoModelForSequenceClassification.from_pretrained(
        base_model, num_labels=1, ignore_mismatched_sizes=True
    )
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    print(f"Base model: {base_model.split('/')[-1]}")
    print(f"Device: {device} | Seed: {seed}")

    # Build training paths
    domain_paths = [domain_train_path] * domain_upsample
    if mmarco_path:
        all_paths = domain_paths + [mmarco_path]
    else:
        all_paths = domain_paths

    train_dataset = PairwiseDataset(all_paths, tokenizer, max_length)
    dev_dataset   = PairwiseDataset([dev_path], tokenizer, max_length)

    domain_n = 2689 * domain_upsample
    mmarco_n = len(train_dataset) - domain_n
    print(f"Train: {len(train_dataset):,} (domain {domain_n:,} | mmarco ~{mmarco_n:,})")
    print(f"Dev:   {len(dev_dataset):,}")

    train_loader = DataLoader(
        train_dataset, batch_size=batch_size,
        shuffle=True, num_workers=0,
        worker_init_fn=lambda w: set_seed(seed + w),
    )
    dev_loader = DataLoader(dev_dataset, batch_size=batch_size, num_workers=0)

    optimizer   = AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    total_steps = len(train_loader) * epochs
    scheduler   = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=int(0.1 * total_steps),
        num_training_steps=total_steps,
    )
    criterion  = StageALoss()
    Path(output_dir).mkdir(parents=True, exist_ok=True)

    best_acc   = 0
    no_improve = 0

    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for batch in train_loader:
            optimizer.zero_grad()
            pos = model(
                input_ids=batch["pos_input_ids"].to(device),
                attention_mask=batch["pos_attention_mask"].to(device),
            ).logits
            neg = model(
                input_ids=batch["neg_input_ids"].to(device),
                attention_mask=batch["neg_attention_mask"].to(device),
            ).logits
            loss = criterion(pos, neg)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            total_loss += loss.item()

        acc = evaluate_pairwise(model, dev_loader, device)
        n   = len(train_loader)
        print(f"Epoch {epoch+1}/{epochs} | Loss: {total_loss/n:.4f} | Dev Acc: {acc:.4f}")

        if acc > best_acc:
            best_acc   = acc
            no_improve = 0
            model.save_pretrained(f"{output_dir}/best")
            tokenizer.save_pretrained(f"{output_dir}/best")
            print(f"  → Saved best (acc={best_acc:.4f})")
        else:
            no_improve += 1
            print(f"  No improve ({no_improve}/{patience})")
            if no_improve >= patience:
                print(f"Early stopping at epoch {epoch+1}")
                break

    print(f"\nStage A done. Best dev acc: {best_acc:.4f}")
    return f"{output_dir}/best"


print("Cell 04 done — train_stage_a loaded ✓")

## Train stage B

In [30]:
def train_stage_b(
    stage_a_checkpoint,
    rerank_path,
    domain_train_path,
    dev_path,
    output_dir,
    epochs=5,
    batch_size=16,
    lr=1e-5,
    max_length=512,
    temperature=2.0,
    alpha=0.7,
    patience=2,
    seed=42,
):
    set_seed(seed)
    reranker_tok = AutoTokenizer.from_pretrained(stage_a_checkpoint)
    model        = AutoModelForSequenceClassification.from_pretrained(stage_a_checkpoint)
    device       = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    print(f"Checkpoint: {stage_a_checkpoint.split('/')[-2]}/{stage_a_checkpoint.split('/')[-1]}")
    print(f"Device: {device} | Seed: {seed} | alpha={alpha} | T={temperature}")

    kd_dataset = ListwiseKDDataset(rerank_path, reranker_tok, max_length)
    kd_loader  = DataLoader(
        kd_dataset, batch_size=batch_size,
        shuffle=True, collate_fn=collate_listwise,
        num_workers=0, worker_init_fn=lambda w: set_seed(seed + w),
    )

    cl_dataset = PairwiseDataset([domain_train_path], reranker_tok, max_length)
    cl_loader  = DataLoader(cl_dataset, batch_size=batch_size * 2, shuffle=True, num_workers=0)
    cl_iter    = iter(cl_loader)

    dev_dataset = PairwiseDataset([dev_path], reranker_tok, max_length)
    dev_loader  = DataLoader(dev_dataset, batch_size=32, num_workers=0)

    optimizer   = AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    total_steps = len(kd_loader) * epochs
    scheduler   = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=int(0.05 * total_steps),
        num_training_steps=total_steps,
    )
    kd_crit = ListwiseKLLoss(temperature=temperature)
    cl_crit = StageALoss()
    Path(output_dir).mkdir(parents=True, exist_ok=True)

    best_acc   = 0
    no_improve = 0

    for epoch in range(epochs):
        model.train()
        total_loss = total_kd = total_cl = 0

        for batch in kd_loader:
            optimizer.zero_grad()
            all_logits = model(
                input_ids=batch["input_ids"].to(device),
                attention_mask=batch["attention_mask"].to(device),
            ).logits.squeeze(-1)

            kd_loss = torch.tensor(0.0, device=device)
            offset  = 0
            for i, size in enumerate(batch["sizes"]):
                kd_loss += kd_crit(
                    all_logits[offset:offset+size].unsqueeze(0),
                    batch["bge_scores"][i].to(device).unsqueeze(0),
                )
                offset += size
            kd_loss /= len(batch["sizes"])

            try:
                cl_batch = next(cl_iter)
            except StopIteration:
                cl_iter  = iter(cl_loader)
                cl_batch = next(cl_iter)

            pos = model(
                input_ids=cl_batch["pos_input_ids"].to(device),
                attention_mask=cl_batch["pos_attention_mask"].to(device),
            ).logits
            neg = model(
                input_ids=cl_batch["neg_input_ids"].to(device),
                attention_mask=cl_batch["neg_attention_mask"].to(device),
            ).logits
            cl_loss = cl_crit(pos, neg)

            loss = alpha * kd_loss + (1 - alpha) * cl_loss
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            total_loss += loss.item()
            total_kd   += kd_loss.item()
            total_cl   += cl_loss.item()

        acc = evaluate_pairwise(model, dev_loader, device)
        n   = len(kd_loader)
        print(f"Epoch {epoch+1}/{epochs} | Loss: {total_loss/n:.4f} | "
              f"KD: {total_kd/n:.4f} | CL: {total_cl/n:.4f} | Dev Acc: {acc:.4f}")

        if acc > best_acc:
            best_acc   = acc
            no_improve = 0
            model.save_pretrained(f"{output_dir}/best")
            reranker_tok.save_pretrained(f"{output_dir}/best")
            print(f"  → Saved best (acc={best_acc:.4f})")
        else:
            no_improve += 1
            print(f"  No improve ({no_improve}/{patience})")
            if no_improve >= patience:
                print(f"Early stopping at epoch {epoch+1}")
                break

    print(f"\nStage B done. Best dev acc: {best_acc:.4f}")
    return f"{output_dir}/best"


print("Cell 05 done — train_stage_b loaded ✓")

Cell 05 done — train_stage_b loaded ✓


## Train MarginMSE 

In [ ]:
def train_margin_mse(
    base_checkpoint,
    train_path,
    dev_path,
    output_dir,
    epochs=5,
    batch_size=16,
    lr=1e-5,
    max_length=512,
    patience=2,
    seed=42,
):
    set_seed(seed)
    tokenizer = AutoTokenizer.from_pretrained(base_checkpoint)
    model     = AutoModelForSequenceClassification.from_pretrained(
        base_checkpoint, num_labels=1, ignore_mismatched_sizes=True
    )
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    print(f"Checkpoint: {base_checkpoint.split('/')[-2]}/{base_checkpoint.split('/')[-1]}")
    print(f"Device: {device} | Seed: {seed}")

    train_dataset = MarginMSEDataset(train_path, tokenizer, max_length)
    dev_dataset   = PairwiseDataset([dev_path], tokenizer, max_length)
    print(f"Train: {len(train_dataset):,} | Dev: {len(dev_dataset):,}")

    train_loader = DataLoader(
        train_dataset, batch_size=batch_size,
        shuffle=True, num_workers=0,
        worker_init_fn=lambda w: set_seed(seed + w),
    )
    dev_loader = DataLoader(dev_dataset, batch_size=batch_size, num_workers=0)

    optimizer   = AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    total_steps = len(train_loader) * epochs
    scheduler   = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=int(0.1 * total_steps),
        num_training_steps=total_steps,
    )
    criterion  = MarginMSELoss()
    Path(output_dir).mkdir(parents=True, exist_ok=True)

    best_acc   = 0
    no_improve = 0

    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for batch in train_loader:
            optimizer.zero_grad()
            pos = model(
                input_ids=batch["pos_input_ids"].to(device),
                attention_mask=batch["pos_attention_mask"].to(device),
            ).logits.squeeze(-1)
            neg = model(
                input_ids=batch["neg_input_ids"].to(device),
                attention_mask=batch["neg_attention_mask"].to(device),
            ).logits.squeeze(-1)
            loss = criterion(pos, neg, batch["teacher_margin"].to(device))
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            total_loss += loss.item()

        acc = evaluate_pairwise(model, dev_loader, device)
        n   = len(train_loader)
        print(f"Epoch {epoch+1}/{epochs} | Loss: {total_loss/n:.4f} | Dev Acc: {acc:.4f}")

        if acc > best_acc:
            best_acc   = acc
            no_improve = 0
            model.save_pretrained(f"{output_dir}/best")
            tokenizer.save_pretrained(f"{output_dir}/best")
            print(f"  → Saved best (acc={best_acc:.4f})")
        else:
            no_improve += 1
            print(f"  No improve ({no_improve}/{patience})")
            if no_improve >= patience:
                print(f"Early stopping at epoch {epoch+1}")
                break

    print(f"\nMargin-MSE done. Best dev acc: {best_acc:.4f}")
    return f"{output_dir}/best"


print("Cell 06 done — train_margin_mse loaded ✓")

## Train RankNet/ADR-MSE

In [4]:
def train_stage_b_ranknet(
    stage_a_checkpoint,
    rerank_path,
    domain_train_path,
    dev_path,
    output_dir,
    loss_type="ranknet",    # "ranknet" hoặc "adr_mse"
    epochs=5,
    batch_size=8,           # batch nhỏ hơn vì listwise nặng hơn
    lr=1e-5,
    max_length=512,
    alpha=0.7,              # KD weight
    patience=2,
    seed=42,
):
    set_seed(seed)
    tok   = AutoTokenizer.from_pretrained(stage_a_checkpoint)
    model = AutoModelForSequenceClassification.from_pretrained(stage_a_checkpoint)
    device = torch.device("cuda")
    model.to(device)
    print(f"Loss: {loss_type} | alpha={alpha} | Seed: {seed}")

    kd_dataset = ListwiseRankDataset(rerank_path, tok, max_length)
    kd_loader  = DataLoader(
        kd_dataset, batch_size=batch_size,
        shuffle=True, collate_fn=collate_listwise_rank,
        num_workers=0, worker_init_fn=lambda w: set_seed(seed + w),
    )

    cl_dataset = PairwiseDataset([domain_train_path], tok, max_length)
    cl_loader  = DataLoader(cl_dataset, batch_size=batch_size*2, shuffle=True, num_workers=0)
    cl_iter    = iter(cl_loader)

    dev_dataset = PairwiseDataset([dev_path], tok, max_length)
    dev_loader  = DataLoader(dev_dataset, batch_size=32, num_workers=0)

    optimizer   = AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    total_steps = len(kd_loader) * epochs
    scheduler   = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=int(0.05 * total_steps),
        num_training_steps=total_steps,
    )

    # Chọn loss function
    if loss_type == "ranknet":
        kd_crit = RankNetSoftLoss()
    elif loss_type == "adr_mse":
        kd_crit = ADRMSELoss()
    else:
        raise ValueError(f"Unknown loss_type: {loss_type}")

    cl_crit = StageALoss()
    Path(output_dir).mkdir(parents=True, exist_ok=True)

    best_acc = 0
    no_improve = 0

    for epoch in range(epochs):
        model.train()
        total_loss = total_kd = total_cl = 0

        for batch in kd_loader:
            optimizer.zero_grad()

            all_logits = model(
                input_ids=batch["input_ids"].to(device),
                attention_mask=batch["attention_mask"].to(device),
            ).logits.squeeze(-1)

            kd_loss = torch.tensor(0.0, device=device)
            offset  = 0
            for i, size in enumerate(batch["sizes"]):
                q_logits = all_logits[offset:offset+size].unsqueeze(0)

                if loss_type == "ranknet":
                    q_scores = batch["bge_scores"][i].to(device).unsqueeze(0)
                    kd_loss += kd_crit(q_logits, q_scores)
                else:  # adr_mse
                    q_ranks  = batch["ranks"][i].to(device).unsqueeze(0)
                    kd_loss += kd_crit(q_logits, q_ranks)

                offset += size
            kd_loss /= len(batch["sizes"])

            try:
                cl_batch = next(cl_iter)
            except StopIteration:
                cl_iter  = iter(cl_loader)
                cl_batch = next(cl_iter)

            pos = model(
                input_ids=cl_batch["pos_input_ids"].to(device),
                attention_mask=cl_batch["pos_attention_mask"].to(device),
            ).logits
            neg = model(
                input_ids=cl_batch["neg_input_ids"].to(device),
                attention_mask=cl_batch["neg_attention_mask"].to(device),
            ).logits
            cl_loss = cl_crit(pos, neg)

            loss = alpha * kd_loss + (1 - alpha) * cl_loss
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            total_loss += loss.item()
            total_kd   += kd_loss.item()
            total_cl   += cl_loss.item()

        acc = evaluate_pairwise(model, dev_loader, device)
        n   = len(kd_loader)
        print(f"Epoch {epoch+1}/{epochs} | Loss: {total_loss/n:.4f} | "
              f"KD({loss_type}): {total_kd/n:.4f} | CL: {total_cl/n:.4f} | Dev Acc: {acc:.4f}")

        if acc > best_acc:
            best_acc   = acc
            no_improve = 0
            model.save_pretrained(f"{output_dir}/best")
            tok.save_pretrained(f"{output_dir}/best")
            print(f"  → Saved best (acc={best_acc:.4f})")
        else:
            no_improve += 1
            print(f"  No improve ({no_improve}/{patience})")
            if no_improve >= patience:
                print(f"Early stopping at epoch {epoch+1}")
                break

    print(f"\n{loss_type} done. Best dev acc: {best_acc:.4f}")
    return f"{output_dir}/best"

## Benchmark Reranker

In [5]:
def benchmark_reranker(
    model,
    tokenizer,
    retrieve_results,
    questions,
    device,
    max_questions=None,
    batch_size=32,
    max_len=512,
    warmup=10,
    label="",
):
    model.eval()
    samples = retrieve_results if max_questions is None else retrieve_results[:max_questions]

    # Warmup
    print(f"\n[{label}] Warming up...")
    for entry in samples[:warmup]:
        pairs = [(entry["question"], c["chunk"]) for c in entry["candidates"]]
        for i in range(0, len(pairs), batch_size):
            b = pairs[i:i+batch_size]
            enc = tokenizer(
                [p[0] for p in b], [p[1] for p in b],
                max_length=max_len, padding=True, truncation=True, return_tensors="pt",
            )
            with torch.no_grad():
                _ = model(
                    input_ids=enc["input_ids"].to(device),
                    attention_mask=enc["attention_mask"].to(device),
                ).logits
    if device.type == "cuda":
        torch.cuda.synchronize()

    # Evaluate
    print(f"[{label}] Evaluating...")
    latencies, recall5, mrr10, total, total_pairs, misses = [], 0, 0, 0, 0, []

    for entry in samples:
        query = entry["question"]
        if query not in questions:
            continue

        candidates  = entry["candidates"]
        gold_ids    = set(questions[query]["gold_chunk_ids"])
        pairs       = [(query, c["chunk"]) for c in candidates]
        total_pairs += len(pairs)
        scores      = []

        if device.type == "cuda":
            torch.cuda.synchronize()
        t0 = time.perf_counter()

        for i in range(0, len(pairs), batch_size):
            b = pairs[i:i+batch_size]
            enc = tokenizer(
                [p[0] for p in b], [p[1] for p in b],
                max_length=max_len, padding=True, truncation=True, return_tensors="pt",
            )
            with torch.no_grad():
                logits = model(
                    input_ids=enc["input_ids"].to(device),
                    attention_mask=enc["attention_mask"].to(device),
                ).logits.squeeze(-1)
            out = logits.detach().cpu().tolist()
            scores.extend([out] if isinstance(out, float) else out)

        if device.type == "cuda":
            torch.cuda.synchronize()
        latencies.append(time.perf_counter() - t0)

        ranked = [c for _, c in sorted(zip(scores, candidates), key=lambda x: -x[0])]
        top5   = [c["chunk_id"] for c in ranked[:5]]

        if any(g in top5 for g in gold_ids):
            recall5 += 1
        else:
            misses.append(query)

        for rank, c in enumerate(ranked[:10], 1):
            if c["chunk_id"] in gold_ids:
                mrr10 += 1.0 / rank
                break

        total += 1

    r5  = recall5 / total
    m10 = mrr10   / total
    lat = np.mean(latencies)
    p50 = np.percentile(latencies, 50)
    p95 = np.percentile(latencies, 95)

    print(f"\n[{label}] RESULTS")
    print("=" * 60)
    print(f"Questions : {total} | Pairs: {total_pairs} | Misses: {len(misses)}")
    print("-" * 60)
    print(f"Recall@5  : {r5:.4f}")
    print(f"MRR@10    : {m10:.4f}")
    print("-" * 60)
    print(f"Latency   : avg={lat*1000:.1f}ms | p50={p50*1000:.1f}ms | p95={p95*1000:.1f}ms")
    print(f"QPS       : {total/sum(latencies):.1f}")

    return {
        "Recall@5": r5, "MRR@10": m10,
        "misses": len(misses), "total": total,
        "avg_latency_ms": lat * 1000,
        "p50_ms": p50 * 1000,
        "p95_ms": p95 * 1000,
        "qps":    total / sum(latencies),
    }


print("Cell 07 done — benchmark_reranker loaded ✓")

Cell 07 done — benchmark_reranker loaded ✓


## Load test data

In [6]:
# Load questions
with open(TEST_Q, encoding="utf-8") as f:
    raw = json.load(f)
questions = {q["question"]: q for q in raw}

# Load retrieve results
retrieve_results = []
with open(RETRIEVE_TEST, encoding="utf-8") as f:
    for line in f:
        retrieve_results.append(json.loads(line))

matched = sum(1 for e in retrieve_results if e["question"] in questions)
print(f"Questions : {len(questions)}")
print(f"Retrieve  : {len(retrieve_results)}")
print(f"Matched   : {matched}")

# Init results dict
ablation_results = {}
print("\nTest data loaded ✓")

Questions : 343
Retrieve  : 343
Matched   : 343

Test data loaded ✓


## MiniLM Base

In [ ]:
tok_base   = AutoTokenizer.from_pretrained(MINILM_BASE)
model_base = AutoModelForSequenceClassification.from_pretrained(MINILM_BASE).to(device).eval()
n_params   = round(sum(p.numel() for p in model_base.parameters()) / 1e6, 1)
print(f"MiniLM base: {n_params}M params")

ablation_results["(1) MiniLM-L12 base"] = benchmark_reranker(
    model=model_base, tokenizer=tok_base,
    retrieve_results=retrieve_results, questions=questions,
    device=device, max_questions=len(retrieve_results),
    batch_size=32, max_len=512,
    label="MiniLM base",
)
ablation_results["(1) MiniLM-L12 base"]["params_M"] = n_params

del model_base, tok_base
torch.cuda.empty_cache()

## Stage A only with ms-macro

In [ ]:
ckpt_a_with_mmarco = train_stage_a(
    domain_train_path=DOMAIN_TRAIN,
    mmarco_path=MMARCO,
    dev_path=DOMAIN_DEV,
    output_dir=CKPT_STAGE_A_WITH_MMARCO,
    domain_upsample=8,
    patience=2,
    epochs=5,
    batch_size=32,
    lr=2e-5,
    seed=42,
)

tok_a   = AutoTokenizer.from_pretrained(ckpt_a_with_mmarco)
model_a = AutoModelForSequenceClassification.from_pretrained(ckpt_a_with_mmarco).to(device).eval()

ablation_results["(2) Stage A only (w/ mMARCO)"] = benchmark_reranker(
    model=model_a, tokenizer=tok_a,
    retrieve_results=retrieve_results, questions=questions,
    device=device, max_questions=len(retrieve_results),
    batch_size=32, max_len=512,
    label="Stage A w/ mMARCO",
)
ablation_results["(2) Stage A only (w/ mMARCO)"]["params_M"] = 33.4

del model_a, tok_a
torch.cuda.empty_cache()

## Stage A only (no ms-macro)

In [ ]:
ckpt_a_no_mmarco = train_stage_a(
    domain_train_path=DOMAIN_TRAIN,
    mmarco_path=None,          
    dev_path=DOMAIN_DEV,
    output_dir=CKPT_STAGE_A_NO_MMARCO,
    domain_upsample=1,        
    patience=2,
    epochs=5,
    batch_size=32,
    lr=2e-5,
    seed=42,
)

tok_a2   = AutoTokenizer.from_pretrained(ckpt_a_no_mmarco)
model_a2 = AutoModelForSequenceClassification.from_pretrained(ckpt_a_no_mmarco).to(device).eval()

ablation_results["(3) Stage A only (no mMARCO)"] = benchmark_reranker(
    model=model_a2, tokenizer=tok_a2,
    retrieve_results=retrieve_results, questions=questions,
    device=device, max_questions=len(retrieve_results),
    batch_size=32, max_len=512,
    label="Stage A no mMARCO",
)
ablation_results["(3) Stage A only (no mMARCO)"]["params_M"] = 33.4

del model_a2, tok_a2
torch.cuda.empty_cache()

## Stage A + Stage B (ListwiseKL)

In [ ]:
ckpt_b_kl = train_stage_b(
    stage_a_checkpoint=ckpt_a_with_mmarco,
    rerank_path=RERANK_991,
    gold_path=GOLD_CHUNKS,
    domain_train_path=DOMAIN_TRAIN,
    dev_path=DOMAIN_DEV,
    output_dir=CKPT_STAGE_B_LISTWISE,
    epochs=5,
    batch_size=16,
    lr=1e-5,
    temperature=2.0,
    alpha=0.7,
    patience=2,
    seed=42,
)

tok_b   = AutoTokenizer.from_pretrained(ckpt_b_kl)
model_b = AutoModelForSequenceClassification.from_pretrained(ckpt_b_kl).to(device).eval()

ablation_results["(4) Stage A+B (ListwiseKL)"] = benchmark_reranker(
    model=model_b, tokenizer=tok_b,
    retrieve_results=retrieve_results, questions=questions,
    device=device, max_questions=len(retrieve_results),
    batch_size=32, max_len=512,
    label="Stage A+B ListwiseKL",
)
ablation_results["(4) Stage A+B (ListwiseKL)"]["params_M"] = 33.4

del model_b, tok_b
torch.cuda.empty_cache()

## Stage A + Stage B (MarginMSE)

In [ ]:
ckpt_b_mse = train_margin_mse(
    base_checkpoint=ckpt_a_with_mmarco,
    train_path=MARGIN_TRAIN,
    dev_path=DOMAIN_DEV,
    output_dir=CKPT_STAGE_B_MARGIN_MSE,
    epochs=5,
    batch_size=16,
    lr=1e-5,
    patience=2,
    seed=42,
)

tok_mse   = AutoTokenizer.from_pretrained(ckpt_b_mse)
model_mse = AutoModelForSequenceClassification.from_pretrained(ckpt_b_mse).to(device).eval()

ablation_results["(5) Stage A+B (Margin-MSE)"] = benchmark_reranker(
    model=model_mse, tokenizer=tok_mse,
    retrieve_results=retrieve_results, questions=questions,
    device=device, max_questions=len(retrieve_results),
    batch_size=32, max_len=512,
    label="Stage A+B Margin-MSE",
)
ablation_results["(5) Stage A+B (Margin-MSE)"]["params_M"] = 33.4

del model_mse, tok_mse
torch.cuda.empty_cache()

## Stage A no macro + Stage B (ListwiseKL)

In [31]:
ckpt_b_kl_nommarco = train_stage_b(
    stage_a_checkpoint=STAGE_A_CKPT,
    rerank_path=RERANK_991,
    domain_train_path=DOMAIN_TRAIN,
    dev_path=DOMAIN_DEV,
    output_dir="/kaggle/working/ablation/stage_b_kl_no_mmarco_991",
    epochs=5, batch_size=16, lr=1e-5,
    temperature=2.0, alpha=0.7,
    patience=2, seed=42,
)

tok_b2   = AutoTokenizer.from_pretrained(ckpt_b_kl_nommarco)
model_b2 = AutoModelForSequenceClassification.from_pretrained(
    ckpt_b_kl_nommarco).to(device).eval()

ablation_results["(6) No mMARCO + Stage B (KL)"] = benchmark_reranker(
    model=model_b2, tokenizer=tok_b2,
    retrieve_results=retrieve_results, questions=questions,
    device=device, max_questions=len(retrieve_results),
    batch_size=32, max_len=512,
    label="No mMARCO + Stage B KL",
)
ablation_results["(6) No mMARCO + Stage B (KL)"]["params_M"] = 33.4

del model_b2, tok_b2
torch.cuda.empty_cache()

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Checkpoint: tqhtqh/stage_a_no_mmarco
Device: cuda | Seed: 42 | alpha=0.7 | T=2.0
KD records: 991
Epoch 1/5 | Loss: 0.5457 | KD: 0.5361 | CL: 0.5679 | Dev Acc: 0.8393


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  → Saved best (acc=0.8393)
Epoch 2/5 | Loss: 0.2076 | KD: 0.0493 | CL: 0.5769 | Dev Acc: 0.8426


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  → Saved best (acc=0.8426)
Epoch 3/5 | Loss: 0.1981 | KD: 0.0480 | CL: 0.5484 | Dev Acc: 0.8295
  No improve (1/2)
Epoch 4/5 | Loss: 0.1953 | KD: 0.0461 | CL: 0.5434 | Dev Acc: 0.8393
  No improve (2/2)
Early stopping at epoch 4

Stage B done. Best dev acc: 0.8426


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]


[No mMARCO + Stage B KL] Warming up...
[No mMARCO + Stage B KL] Evaluating...

[No mMARCO + Stage B KL] RESULTS
Questions : 343 | Pairs: 6860 | Misses: 46
------------------------------------------------------------
Recall@5  : 0.8659
MRR@10    : 0.7226
------------------------------------------------------------
Latency   : avg=22.5ms | p50=22.5ms | p95=23.2ms
QPS       : 44.4


## Stage A no macro + Stage B (MarginMSE)

In [ ]:
ckpt_b_mse_nommarco = train_margin_mse(
    base_checkpoint=ckpt_a_no_mmarco,
    train_path=MARGIN_TRAIN,
    dev_path=DOMAIN_DEV,
    output_dir="/kaggle/working/ablation/stage_b_mse_no_mmarco",
    epochs=5, batch_size=16, lr=1e-5,
    patience=2, seed=42,
)

tok_b3   = AutoTokenizer.from_pretrained(ckpt_b_mse_nommarco)
model_b3 = AutoModelForSequenceClassification.from_pretrained(
    ckpt_b_mse_nommarco).to(device).eval()

ablation_results["(7) No mMARCO + Stage B (MSE)"] = benchmark_reranker(
    model=model_b3, tokenizer=tok_b3,
    retrieve_results=retrieve_results, questions=questions,
    device=device, max_questions=len(retrieve_results),
    batch_size=32, max_len=512,
    label="No mMARCO + Stage B MSE",
)
ablation_results["(7) No mMARCO + Stage B (MSE)"]["params_M"] = 33.4

del model_b3, tok_b3
torch.cuda.empty_cache()

## KD thẳng từ base model - không dùng ms-macro

In [ ]:
ckpt_kd_direct = train_stage_b(
    stage_a_checkpoint=MINILM_BASE,
    rerank_path=RERANK_991,
    gold_path=GOLD_CHUNKS,
    domain_train_path=DOMAIN_TRAIN,
    dev_path=DOMAIN_DEV,
    output_dir="/kaggle/working/ablation/kd_direct_from_base",
    epochs=5, batch_size=16, lr=1e-5,
    temperature=2.0, alpha=1.0,
    patience=2, seed=42,
)

tok_kd   = AutoTokenizer.from_pretrained(ckpt_kd_direct)
model_kd = AutoModelForSequenceClassification.from_pretrained(
    ckpt_kd_direct).to(device).eval()

ablation_results["(8) KD direct (no Stage A)"] = benchmark_reranker(
    model=model_kd, tokenizer=tok_kd,
    retrieve_results=retrieve_results, questions=questions,
    device=device, max_questions=len(retrieve_results),
    batch_size=32, max_len=512,
    label="KD direct from base",
)
ablation_results["(8) KD direct (no Stage A)"]["params_M"] = 33.4
del model_kd, tok_kd
torch.cuda.empty_cache()

## KD từ Stage A checkpoint (no ms-macro)

In [ ]:
ckpt_kd_direct = train_stage_b(
    stage_a_checkpoint="/kaggle/input/datasets/tranquanghuy2809/tqhtqh/stage_a_no_mmarco",
    rerank_path=RERANK_991,
    gold_path=GOLD_CHUNKS,
    domain_train_path=DOMAIN_TRAIN,
    dev_path=DOMAIN_DEV,
    output_dir="/kaggle/working/ablation/kd_direct_from_base",
    epochs=5, batch_size=16, lr=1e-5,
    temperature=2.0, alpha=1.0,
    patience=2, seed=42,
)

tok_kd   = AutoTokenizer.from_pretrained(ckpt_kd_direct)
model_kd = AutoModelForSequenceClassification.from_pretrained(
    ckpt_kd_direct).to(device).eval()

ablation_results["(8) KD direct (no Stage A)"] = benchmark_reranker(
    model=model_kd, tokenizer=tok_kd,
    retrieve_results=retrieve_results, questions=questions,
    device=device, max_questions=len(retrieve_results),
    batch_size=32, max_len=512,
    label="KD direct from base",
)
ablation_results["(8) KD direct (no Stage A)"]["params_M"] = 33.4
del model_kd, tok_kd
torch.cuda.empty_cache()

## Stage A no macro + Stage B (RankNet)

In [7]:
ckpt_ranknet = train_stage_b_ranknet(
    stage_a_checkpoint=STAGE_A_CKPT,
    rerank_path=RERANK_991,
    domain_train_path=DOMAIN_TRAIN, 
    dev_path=DOMAIN_DEV,
    output_dir="/kaggle/working/ablation/stage_b_ranknet",
    loss_type="ranknet",
    epochs=5, batch_size=8, lr=1e-5, alpha=0.7, patience=2, seed=42,
)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loss: ranknet | alpha=0.7 | Seed: 42
Total records: 991
Epoch 1/5 | Loss: 0.7247 | KD(ranknet): 0.8293 | CL: 0.4808 | Dev Acc: 0.8426


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  → Saved best (acc=0.8426)
Epoch 2/5 | Loss: 0.6470 | KD(ranknet): 0.7195 | CL: 0.4778 | Dev Acc: 0.8459


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  → Saved best (acc=0.8459)
Epoch 3/5 | Loss: 0.6308 | KD(ranknet): 0.7216 | CL: 0.4190 | Dev Acc: 0.8459
  No improve (1/2)
Epoch 4/5 | Loss: 0.6244 | KD(ranknet): 0.7222 | CL: 0.3964 | Dev Acc: 0.8623


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  → Saved best (acc=0.8623)
Epoch 5/5 | Loss: 0.6182 | KD(ranknet): 0.7213 | CL: 0.3777 | Dev Acc: 0.8525
  No improve (1/2)

ranknet done. Best dev acc: 0.8623


In [8]:
ckpt_ranknet = "/kaggle/input/datasets/tranquanghuy2809/tqhtqh/stage_b_ranknet"

tok_kd   = AutoTokenizer.from_pretrained(ckpt_ranknet)
model_kd = AutoModelForSequenceClassification.from_pretrained(
    ckpt_ranknet).to(device).eval()

ablation_results["RankNet"] = benchmark_reranker(
    model=model_kd, tokenizer=tok_kd,
    retrieve_results=retrieve_results, questions=questions,
    device=device, max_questions=len(retrieve_results),
    batch_size=32, max_len=512,
    label="RankNet",
)
ablation_results["RankNet"]["params_M"] = 33.4

del model_kd, tok_kd
torch.cuda.empty_cache()

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]


[RankNet] Warming up...
[RankNet] Evaluating...

[RankNet] RESULTS
Questions : 343 | Pairs: 6860 | Misses: 44
------------------------------------------------------------
Recall@5  : 0.8717
MRR@10    : 0.7419
------------------------------------------------------------
Latency   : avg=22.8ms | p50=22.9ms | p95=23.6ms
QPS       : 43.8


## Stage A no macro + Stage B (ADR-MSE)

In [12]:
ckpt_adrmse = train_stage_b_ranknet(
    stage_a_checkpoint=STAGE_A_CKPT,
    rerank_path=RERANK_991,
    domain_train_path=DOMAIN_TRAIN,
    dev_path=DOMAIN_DEV,
    output_dir="/kaggle/working/ablation/stage_b_adrmse",
    loss_type="adr_mse",
    epochs=5, batch_size=8, lr=1e-5, alpha=0.7, patience=2, seed=42,
)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loss: adr_mse | alpha=0.7 | Seed: 42
Total records: 991
Epoch 1/5 | Loss: 0.6648 | KD(adr_mse): 0.8075 | CL: 0.3318 | Dev Acc: 0.8459


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  → Saved best (acc=0.8459)
Epoch 2/5 | Loss: 0.6178 | KD(adr_mse): 0.7356 | CL: 0.3427 | Dev Acc: 0.8459
  No improve (1/2)
Epoch 3/5 | Loss: 0.5795 | KD(adr_mse): 0.7012 | CL: 0.2954 | Dev Acc: 0.8492


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  → Saved best (acc=0.8492)
Epoch 4/5 | Loss: 0.5700 | KD(adr_mse): 0.6879 | CL: 0.2951 | Dev Acc: 0.8492
  No improve (1/2)
Epoch 5/5 | Loss: 0.5566 | KD(adr_mse): 0.6756 | CL: 0.2789 | Dev Acc: 0.8492
  No improve (2/2)
Early stopping at epoch 5

adr_mse done. Best dev acc: 0.8492


In [7]:
ckpt_adrmse = train_stage_b_ranknet(
    stage_a_checkpoint=STAGE_A_CKPT,
    rerank_path=RERANK_991,
    domain_train_path=DOMAIN_TRAIN,
    dev_path=DOMAIN_DEV,
    output_dir="/kaggle/working/ablation/stage_b_adrmse",
    loss_type="adr_mse",
    epochs=5, batch_size=8, lr=1e-5, alpha=1, patience=2, seed=42,
)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loss: adr_mse | alpha=1 | Seed: 42
Total records: 991
Epoch 1/5 | Loss: 0.7944 | KD(adr_mse): 0.7944 | CL: 0.5826 | Dev Acc: 0.8525


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  → Saved best (acc=0.8525)
Epoch 2/5 | Loss: 0.7190 | KD(adr_mse): 0.7190 | CL: 0.7164 | Dev Acc: 0.8459
  No improve (1/2)
Epoch 3/5 | Loss: 0.6798 | KD(adr_mse): 0.6798 | CL: 0.6304 | Dev Acc: 0.8426
  No improve (2/2)
Early stopping at epoch 3

adr_mse done. Best dev acc: 0.8525


In [8]:
tok_kd   = AutoTokenizer.from_pretrained(ckpt_adrmse)
model_kd = AutoModelForSequenceClassification.from_pretrained(
    ckpt_adrmse).to(device).eval()

ablation_results["ADR-MSE"] = benchmark_reranker(
    model=model_kd, tokenizer=tok_kd,
    retrieve_results=retrieve_results, questions=questions,
    device=device, max_questions=len(retrieve_results),
    batch_size=32, max_len=512,
    label="ADR-MSE",
)
ablation_results["ADR-MSE"]["params_M"] = 33.4

del model_kd, tok_kd
torch.cuda.empty_cache()

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]


[ADR-MSE] Warming up...
[ADR-MSE] Evaluating...

[ADR-MSE] RESULTS
Questions : 343 | Pairs: 6860 | Misses: 33
------------------------------------------------------------
Recall@5  : 0.9038
MRR@10    : 0.7529
------------------------------------------------------------
Latency   : avg=22.5ms | p50=22.5ms | p95=23.2ms
QPS       : 44.5


In [11]:
import math

def compute_extended_metrics(
    model, tokenizer, retrieve_results, questions,
    device, batch_size=32, max_len=512, warmup=10, label=""
):
    """Compute Recall@5, MRR@10, NDCG@10, MAP@10, Hit@1"""
    model.eval()

    # Warmup
    for entry in retrieve_results[:warmup]:
        pairs = [(entry["question"], c["chunk"]) for c in entry["candidates"]]
        for i in range(0, len(pairs), batch_size):
            b = pairs[i:i+batch_size]
            enc = tokenizer(
                [p[0] for p in b], [p[1] for p in b],
                max_length=max_len, padding=True,
                truncation=True, return_tensors="pt",
            )
            with torch.no_grad():
                _ = model(
                    input_ids=enc["input_ids"].to(device),
                    attention_mask=enc["attention_mask"].to(device),
                ).logits
    if device.type == "cuda":
        torch.cuda.synchronize()

    recall5_sum = mrr10_sum = ndcg10_sum = map10_sum = hit1_sum = 0
    total = 0

    for entry in retrieve_results:
        query = entry["question"]
        if query not in questions:
            continue

        candidates = entry["candidates"]
        gold_ids   = set(questions[query]["gold_chunk_ids"])
        pairs      = [(query, c["chunk"]) for c in candidates]
        scores     = []

        if device.type == "cuda":
            torch.cuda.synchronize()

        for i in range(0, len(pairs), batch_size):
            b = pairs[i:i+batch_size]
            enc = tokenizer(
                [p[0] for p in b], [p[1] for p in b],
                max_length=max_len, padding=True,
                truncation=True, return_tensors="pt",
            )
            with torch.no_grad():
                logits = model(
                    input_ids=enc["input_ids"].to(device),
                    attention_mask=enc["attention_mask"].to(device),
                ).logits.squeeze(-1)
            out = logits.detach().cpu().tolist()
            scores.extend([out] if isinstance(out, float) else out)

        if device.type == "cuda":
            torch.cuda.synchronize()

        ranked = [c for _, c in sorted(zip(scores, candidates), key=lambda x: -x[0])]

        # Binary relevance list (top-10) 
        rel = [1 if c["chunk_id"] in gold_ids else 0 for c in ranked[:10]]

        # Recall@5 
        top5 = [c["chunk_id"] for c in ranked[:5]]
        if any(g in top5 for g in gold_ids):
            recall5_sum += 1

        # Hit@1 
        if ranked[0]["chunk_id"] in gold_ids:
            hit1_sum += 1

        # MRR@10 
        for rank, c in enumerate(ranked[:10], 1):
            if c["chunk_id"] in gold_ids:
                mrr10_sum += 1.0 / rank
                break

        # NDCG@10 
        # DCG
        dcg = sum(r / math.log2(i + 2) for i, r in enumerate(rel))
        # Ideal DCG: best possible (all gold at top)
        ideal = sorted(rel, reverse=True)
        idcg  = sum(r / math.log2(i + 2) for i, r in enumerate(ideal))
        ndcg10_sum += (dcg / idcg) if idcg > 0 else 0.0

        # MAP@10 
        ap = 0.0
        hits = 0
        for i, r in enumerate(rel, 1):
            if r == 1:
                hits += 1
                ap   += hits / i
        n_gold_in_top10 = sum(rel)
        map10_sum += (ap / n_gold_in_top10) if n_gold_in_top10 > 0 else 0.0

        total += 1

    return {
        "Hit@1":     hit1_sum    / total,
        "Recall@5":  recall5_sum / total,
        "MRR@10":    mrr10_sum   / total,
        "NDCG@10":   ndcg10_sum  / total,
        "MAP@10":    map10_sum   / total,
        "total":     total,
    }

In [12]:
import math

NEW_MODELS = {
    # "(8) No mMARCO + RankNet": ("/kaggle/input/datasets/tranquanghuy2809/tqhtqh/stage_b_ranknet", 512),
    "(9) No mMARCO + ADR-MSE": ("/kaggle/working/ablation/stage_b_adrmse/best",  512),
}

for name, (ckpt, max_len) in NEW_MODELS.items():
    print(f"\n{'='*60}")
    print(f"Evaluating: {name}")
    print(f"{'='*60}")

    tok   = AutoTokenizer.from_pretrained(ckpt)
    model = AutoModelForSequenceClassification.from_pretrained(ckpt).to(device).eval()
    n_params = round(sum(p.numel() for p in model.parameters()) / 1e6, 1)
    print(f"Loaded {n_params}M params")

    result = compute_extended_metrics(
        model=model, tokenizer=tok,
        retrieve_results=retrieve_results, questions=questions,
        device=device, batch_size=32, max_len=max_len,
        label=name,
    )
    result["params_M"] = n_params
    extended_results[name] = result

    print(f"  Hit@1    = {result['Hit@1']:.4f}")
    print(f"  Recall@5 = {result['Recall@5']:.4f}")
    print(f"  MRR@10   = {result['MRR@10']:.4f}")
    print(f"  NDCG@10  = {result['NDCG@10']:.4f}")
    print(f"  MAP@10   = {result['MAP@10']:.4f}")

    del model, tok
    torch.cuda.empty_cache()

# ── In bảng so sánh nhanh ─────────────────────────────────────────
print(f"\n{'='*80}")
print(f"{'Model':<35} {'Hit@1':>8} {'Recall@5':>10} {'MRR@10':>10} {'NDCG@10':>10}")
print("-"*80)
for name in ["(6) No mMARCO + B (KL)", "(7) No mMARCO + B (MSE)",
             "(8) No mMARCO + RankNet", "(9) No mMARCO + ADR-MSE",
             "PhoRanker", "BGE-M3 (teacher)"]:
    if name not in extended_results:
        continue
    r = extended_results[name]
    print(f"{name:<35} {r['Hit@1']:>8.4f} {r['Recall@5']:>10.4f} "
          f"{r['MRR@10']:>10.4f} {r['NDCG@10']:>10.4f}")
print("="*80)


Evaluating: (9) No mMARCO + ADR-MSE


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loaded 33.4M params
  Hit@1    = 0.6501
  Recall@5 = 0.9038
  MRR@10   = 0.7529
  NDCG@10  = 0.7960
  MAP@10   = 0.7461

Model                                  Hit@1   Recall@5     MRR@10    NDCG@10
--------------------------------------------------------------------------------
(9) No mMARCO + ADR-MSE               0.6501     0.9038     0.7529     0.7960


## PhoRanker

In [ ]:
import warnings, transformers
transformers.logging.set_verbosity_error()
warnings.filterwarnings("ignore")

tok_pho   = AutoTokenizer.from_pretrained(PHORANKER, use_fast=False)
model_pho = AutoModelForSequenceClassification.from_pretrained(PHORANKER).to(device).eval()
model_pho = model_pho.half()  # fp16
n_params  = round(sum(p.numel() for p in model_pho.parameters()) / 1e6, 1)
print(f"PhoRanker: {n_params}M params")

# PhoRanker: pre-truncate text to avoid warning spam
# Custom wrapper với max_chars để tránh warning
class TruncatedTokenizer:
    def __init__(self, tok, max_chars_q=500, max_chars_d=1000):
        self.tok     = tok
        self.max_q   = max_chars_q
        self.max_d   = max_chars_d

    def __call__(self, queries, docs, **kwargs):
        q_trunc = [q[:self.max_q] for q in queries]
        d_trunc = [d[:self.max_d] for d in docs]
        return self.tok(q_trunc, d_trunc, **kwargs)

    def __getattr__(self, name):
        return getattr(self.tok, name)

tok_pho_wrapped = TruncatedTokenizer(tok_pho)

ablation_results["PhoRanker (VI baseline)"] = benchmark_reranker(
    model=model_pho, tokenizer=tok_pho_wrapped,
    retrieve_results=retrieve_results, questions=questions,
    device=device, max_questions=len(retrieve_results),
    batch_size=32, max_len=256,
    label="PhoRanker",
)
ablation_results["PhoRanker (VI baseline)"]["params_M"] = n_params

del model_pho, tok_pho
torch.cuda.empty_cache()

## Bge-reranker-v2-m3

In [ ]:
tok_bge   = AutoTokenizer.from_pretrained(BGE_DIR)
model_bge = AutoModelForSequenceClassification.from_pretrained(BGE_DIR).to(device).eval()
n_params  = round(sum(p.numel() for p in model_bge.parameters()) / 1e6, 1)
print(f"BGE-M3: {n_params}M params")

ablation_results["BGE-M3 (teacher)"] = benchmark_reranker(
    model=model_bge, tokenizer=tok_bge,
    retrieve_results=retrieve_results, questions=questions,
    device=device, max_questions=len(retrieve_results),
    batch_size=32, max_len=512,
    label="BGE-M3",
)
ablation_results["BGE-M3 (teacher)"]["params_M"] = n_params

del model_bge, tok_bge
torch.cuda.empty_cache()

## Benchmark

In [10]:
import math

def compute_extended_metrics(
    model, tokenizer, retrieve_results, questions,
    device, batch_size=32, max_len=512, warmup=10, label=""
):
    """Compute Recall@5, MRR@10, NDCG@10, MAP@10, Hit@1"""
    model.eval()

    # Warmup
    for entry in retrieve_results[:warmup]:
        pairs = [(entry["question"], c["chunk"]) for c in entry["candidates"]]
        for i in range(0, len(pairs), batch_size):
            b = pairs[i:i+batch_size]
            enc = tokenizer(
                [p[0] for p in b], [p[1] for p in b],
                max_length=max_len, padding=True,
                truncation=True, return_tensors="pt",
            )
            with torch.no_grad():
                _ = model(
                    input_ids=enc["input_ids"].to(device),
                    attention_mask=enc["attention_mask"].to(device),
                ).logits
    if device.type == "cuda":
        torch.cuda.synchronize()

    recall5_sum = mrr10_sum = ndcg10_sum = map10_sum = hit1_sum = 0
    total = 0

    for entry in retrieve_results:
        query = entry["question"]
        if query not in questions:
            continue

        candidates = entry["candidates"]
        gold_ids   = set(questions[query]["gold_chunk_ids"])
        pairs      = [(query, c["chunk"]) for c in candidates]
        scores     = []

        if device.type == "cuda":
            torch.cuda.synchronize()

        for i in range(0, len(pairs), batch_size):
            b = pairs[i:i+batch_size]
            enc = tokenizer(
                [p[0] for p in b], [p[1] for p in b],
                max_length=max_len, padding=True,
                truncation=True, return_tensors="pt",
            )
            with torch.no_grad():
                logits = model(
                    input_ids=enc["input_ids"].to(device),
                    attention_mask=enc["attention_mask"].to(device),
                ).logits.squeeze(-1)
            out = logits.detach().cpu().tolist()
            scores.extend([out] if isinstance(out, float) else out)

        if device.type == "cuda":
            torch.cuda.synchronize()

        ranked = [c for _, c in sorted(zip(scores, candidates), key=lambda x: -x[0])]

        # Binary relevance list (top-10) 
        rel = [1 if c["chunk_id"] in gold_ids else 0 for c in ranked[:10]]

        # Recall@5 
        top5 = [c["chunk_id"] for c in ranked[:5]]
        if any(g in top5 for g in gold_ids):
            recall5_sum += 1

        # Hit@1 
        if ranked[0]["chunk_id"] in gold_ids:
            hit1_sum += 1

        # MRR@10 
        for rank, c in enumerate(ranked[:10], 1):
            if c["chunk_id"] in gold_ids:
                mrr10_sum += 1.0 / rank
                break

        # NDCG@10 
        # DCG
        dcg = sum(r / math.log2(i + 2) for i, r in enumerate(rel))
        # Ideal DCG: best possible (all gold at top)
        ideal = sorted(rel, reverse=True)
        idcg  = sum(r / math.log2(i + 2) for i, r in enumerate(ideal))
        ndcg10_sum += (dcg / idcg) if idcg > 0 else 0.0

        # MAP@10 
        ap = 0.0
        hits = 0
        for i, r in enumerate(rel, 1):
            if r == 1:
                hits += 1
                ap   += hits / i
        n_gold_in_top10 = sum(rel)
        map10_sum += (ap / n_gold_in_top10) if n_gold_in_top10 > 0 else 0.0

        total += 1

    return {
        "Hit@1":     hit1_sum    / total,
        "Recall@5":  recall5_sum / total,
        "MRR@10":    mrr10_sum   / total,
        "NDCG@10":   ndcg10_sum  / total,
        "MAP@10":    map10_sum   / total,
        "total":     total,
    }


EVAL_MODELS = {
    "(1) MiniLM-L12 base":           (MINILM_BASE,                                              512,  False),
    "(2) Stage A (w/ mMARCO)":       (CKPT_STAGE_A_WITH_MMARCO + "/best",                       512,  False),
    "(3) Stage A (no mMARCO)":       (CKPT_STAGE_A_NO_MMARCO   + "/best",                       512,  False),
    "(4) Stage A+B (KL)":            (CKPT_STAGE_B_LISTWISE     + "/best",                       512,  False),
    "(5) Stage A+B (Margin-MSE)":    (CKPT_STAGE_B_MARGIN_MSE   + "/best",                       512,  False),
    "(6) No mMARCO + B (KL)":        ("/kaggle/working/ablation/stage_b_kl_no_mmarco/best",      512,  False),
    "(7) No mMARCO + B (MSE)":       ("/kaggle/working/ablation/stage_b_mse_no_mmarco/best",     512,  False),
    "PhoRanker":                     (PHORANKER,                                                 256,  True),
    "BGE-M3 (teacher)":              (BGE_DIR,                                                   512,  False),
}

extended_results = {}

for name, (ckpt, max_len, is_phoranker) in EVAL_MODELS.items():
    print(f"\n{'='*60}")
    print(f"Evaluating: {name}")
    print(f"{'='*60}")

    try:
        if is_phoranker:
            import warnings, transformers
            transformers.logging.set_verbosity_error()
            warnings.filterwarnings("ignore")
            tok = TruncatedTokenizer(
                AutoTokenizer.from_pretrained(ckpt, use_fast=False)
            )
            model = AutoModelForSequenceClassification.from_pretrained(ckpt).to(device).eval()
            model = model.half()
        else:
            tok   = AutoTokenizer.from_pretrained(ckpt)
            model = AutoModelForSequenceClassification.from_pretrained(ckpt).to(device).eval()

        n_params = round(sum(p.numel() for p in model.parameters()) / 1e6, 1)

        result = compute_extended_metrics(
            model=model, tokenizer=tok,
            retrieve_results=retrieve_results, questions=questions,
            device=device, batch_size=32, max_len=max_len,
            label=name,
        )
        result["params_M"] = n_params
        extended_results[name] = result
        print(f"  Hit@1={result['Hit@1']:.4f} | Recall@5={result['Recall@5']:.4f} | "
              f"MRR@10={result['MRR@10']:.4f} | NDCG@10={result['NDCG@10']:.4f} | "
              f"MAP@10={result['MAP@10']:.4f}")

    except Exception as e:
        print(f"  ERROR: {e}")

    finally:
        if 'model' in dir():
            del model
        if 'tok' in dir():
            del tok
        torch.cuda.empty_cache()


# ── Final table ───────────────────────────────────────────────────
print("\n\n" + "=" * 100)
print("FULL METRICS TABLE")
print("=" * 100)
print(f"{'Model':<35} {'Params':>7} {'Hit@1':>8} {'Recall@5':>10} "
      f"{'MRR@10':>10} {'NDCG@10':>10} {'MAP@10':>10}")
print("-" * 100)

order = [
    "(1) MiniLM-L12 base",
    "(2) Stage A (w/ mMARCO)",
    "(3) Stage A (no mMARCO)",
    "(4) Stage A+B (KL)",
    "(5) Stage A+B (Margin-MSE)",
    "(6) No mMARCO + B (KL)",
    "(7) No mMARCO + B (MSE)",
    "PhoRanker",
    "BGE-M3 (teacher)",
]

for name in order:
    if name not in extended_results:
        continue
    r = extended_results[name]
    print(
        f"{name:<35} "
        f"{r.get('params_M', 0):>6.1f}M "
        f"{r['Hit@1']:>8.4f} "
        f"{r['Recall@5']:>10.4f} "
        f"{r['MRR@10']:>10.4f} "
        f"{r['NDCG@10']:>10.4f} "
        f"{r['MAP@10']:>10.4f}"
    )

print("=" * 100)


Evaluating: (1) MiniLM-L12 base


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: /kaggle/input/datasets/tranquanghuy2809/tqhtqh/ms-marco-MiniLM-L12-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  Hit@1=0.4694 | Recall@5=0.7347 | MRR@10=0.5869 | NDCG@10=0.6536 | MAP@10=0.5823

Evaluating: (2) Stage A (w/ mMARCO)
  ERROR: Repo id must be in the form 'repo_name' or 'namespace/repo_name': '/kaggle/working/ablation/stage_a_with_mmarco/best'. Use `repo_type` argument if needed.

Evaluating: (3) Stage A (no mMARCO)
  ERROR: Repo id must be in the form 'repo_name' or 'namespace/repo_name': '/kaggle/working/ablation/stage_a_no_mmarco/best'. Use `repo_type` argument if needed.

Evaluating: (4) Stage A+B (KL)
  ERROR: Repo id must be in the form 'repo_name' or 'namespace/repo_name': '/kaggle/working/ablation/stage_b_listwise/best'. Use `repo_type` argument if needed.

Evaluating: (5) Stage A+B (Margin-MSE)
  ERROR: Repo id must be in the form 'repo_name' or 'namespace/repo_name': '/kaggle/working/ablation/stage_b_margin_mse/best'. Use `repo_type` argument if needed.

Evaluating: (6) No mMARCO + B (KL)
  ERROR: Repo id must be in the form 'repo_name' or 'namespace/repo_name': '/kaggle/wo

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
import os, shutil
from pathlib import Path

# ── Danh sách checkpoints cần nén ─────────────────────────────────
CHECKPOINTS = {
    "stage_a_with_mmarco":   CKPT_STAGE_A_WITH_MMARCO + "/best",
    "stage_a_no_mmarco":     CKPT_STAGE_A_NO_MMARCO   + "/best",
    "stage_b_listwise":      CKPT_STAGE_B_LISTWISE     + "/best",
    "stage_b_margin_mse":    CKPT_STAGE_B_MARGIN_MSE   + "/best",
    "stage_b_kl_no_mmarco":  "/kaggle/working/ablation/stage_b_kl_no_mmarco/best",
    "stage_b_mse_no_mmarco": "/kaggle/working/ablation/stage_b_mse_no_mmarco/best",
}

OUTPUT_DIR = "/kaggle/working/checkpoints_export"
os.makedirs(OUTPUT_DIR, exist_ok=True)

for name, ckpt_path in CHECKPOINTS.items():
    if not os.path.exists(ckpt_path):
        print(f"⚠ Skipping {name} — path not found: {ckpt_path}")
        continue

    zip_path = f"{OUTPUT_DIR}/{name}"
    print(f"Packing {name}...")
    shutil.make_archive(zip_path, "zip", ckpt_path)
    
    size_mb = os.path.getsize(zip_path + ".zip") / 1024**2
    print(f"  → {zip_path}.zip ({size_mb:.1f} MB)")

print("\nDone! Files in export dir:")
for f in sorted(os.listdir(OUTPUT_DIR)):
    size = os.path.getsize(f"{OUTPUT_DIR}/{f}") / 1024**2
    print(f"  {f:<45} {size:>8.1f} MB")

In [ ]:
import json

INPUT  = "/kaggle/working/domain_train_with_teacher_scores.jsonl"
OUTPUT = "/kaggle/working/domain_train_margin_filtered.jsonl"

kept = removed_neg_margin = removed_giveaway = 0

with open(INPUT) as fin, open(OUTPUT, "w") as fout:
    for line in fin:
        d = json.loads(line)
        
        # Filter 1: Teacher margin phải dương
        if d["teacher_margin"] <= 0.1:
            removed_neg_margin += 1
            continue
        
        # Filter 2: Negative không chứa query (giveaway check)
        query_lower = d["query"].lower().strip()
        neg_lower   = d["negative"].lower()
        
        # Nếu negative chứa >50% từ của query → giveaway
        query_words = set(query_lower.split())
        neg_words   = set(neg_lower.split())
        overlap     = len(query_words & neg_words) / max(len(query_words), 1)
        
        if overlap > 0.7:  # giveaway threshold
            removed_giveaway += 1
            continue
        
        # Filter 3: Negative không quá giống positive (literal text match)
        if d["negative"][:200] == d["positive"][:200]:
            removed_giveaway += 1
            continue
        
        fout.write(line)
        kept += 1

print(f"Kept: {kept}")
print(f"Removed (neg margin):  {removed_neg_margin}")
print(f"Removed (giveaway):    {removed_giveaway}")
print(f"Total removed: {removed_neg_margin + removed_giveaway}")

In [ ]:
import numpy as np

margins = []
with open(OUTPUT) as f:
    for line in f:
        d = json.loads(line)
        margins.append(d["teacher_margin"])

margins = np.array(margins)
print(f"\nAfter filter:")
print(f"Total: {len(margins)}")
print(f"Mean margin: {margins.mean():.4f}")
print(f"Min margin:  {margins.min():.4f}")
print(f"Max margin:  {margins.max():.4f}")
print(f"Negative margins: {(margins < 0).sum()}")

In [ ]:
# ── MarginMSE Dataset ────────────────────────────────────────────
class MarginMSEDataset(Dataset):
    def __init__(self, path, tokenizer, max_length=512):
        self.data = []
        with open(path) as f:
            for line in f:
                self.data.append(json.loads(line))
        self.tok     = tokenizer
        self.max_len = max_length

    def encode(self, query, passage):
        return self.tok(
            query, passage,
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )

    def __getitem__(self, idx):
        d   = self.data[idx]
        pos = self.encode(d["query"], d["positive"])
        neg = self.encode(d["query"], d["negative"])
        return {
            "pos_input_ids":      pos["input_ids"].squeeze(),
            "pos_attention_mask": pos["attention_mask"].squeeze(),
            "neg_input_ids":      neg["input_ids"].squeeze(),
            "neg_attention_mask": neg["attention_mask"].squeeze(),
            "teacher_margin":     torch.tensor(d["teacher_margin"], dtype=torch.float),
        }

    def __len__(self):
        return len(self.data)


# ── Margin-MSE Loss ──────────────────────────────────────────────
class MarginMSELoss(nn.Module):
    """Hofstätter et al. 2020 — student margin matches teacher margin"""
    def __init__(self):
        super().__init__()
        self.mse = nn.MSELoss()

    def forward(self, student_pos, student_neg, teacher_margin):
        student_margin = student_pos - student_neg
        return self.mse(student_margin, teacher_margin)


# ── Train function với Margin-MSE ────────────────────────────────
def train_margin_mse(
    base_checkpoint,
    train_path,
    dev_path,
    output_dir,
    epochs=5,
    batch_size=16,
    lr=1e-5,
    max_length=512,
    patience=2,
    seed=42,
):
    set_seed(seed)

    tokenizer = AutoTokenizer.from_pretrained(base_checkpoint)
    model     = AutoModelForSequenceClassification.from_pretrained(
        base_checkpoint, num_labels=1, ignore_mismatched_sizes=True
    )
    device = torch.device("cuda")
    model.to(device)
    print(f"Device: {device} | Seed: {seed}")

    train_dataset = MarginMSEDataset(train_path, tokenizer, max_length)
    dev_dataset   = PairwiseDataset([dev_path], tokenizer, max_length)
    print(f"Train: {len(train_dataset):,} | Dev: {len(dev_dataset):,}")

    train_loader = DataLoader(train_dataset, batch_size=batch_size,
                              shuffle=True, num_workers=0,
                              worker_init_fn=lambda w: set_seed(seed + w))
    dev_loader   = DataLoader(dev_dataset, batch_size=batch_size, num_workers=0)

    optimizer   = AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    total_steps = len(train_loader) * epochs
    scheduler   = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=int(0.1 * total_steps),
        num_training_steps=total_steps
    )
    criterion = MarginMSELoss()
    Path(output_dir).mkdir(parents=True, exist_ok=True)

    best_acc   = 0
    no_improve = 0

    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for batch in train_loader:
            optimizer.zero_grad()
            pos_logits = model(
                input_ids=batch["pos_input_ids"].to(device),
                attention_mask=batch["pos_attention_mask"].to(device)
            ).logits.squeeze(-1)
            neg_logits = model(
                input_ids=batch["neg_input_ids"].to(device),
                attention_mask=batch["neg_attention_mask"].to(device)
            ).logits.squeeze(-1)

            loss = criterion(pos_logits, neg_logits,
                             batch["teacher_margin"].to(device))
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            total_loss += loss.item()

        acc = evaluate(model, dev_loader, device)
        print(f"Epoch {epoch+1}/{epochs} | Loss: {total_loss/len(train_loader):.4f} | Domain Acc: {acc:.4f}")

        if acc > best_acc:
            best_acc   = acc
            no_improve = 0
            model.save_pretrained(f"{output_dir}/best")
            tokenizer.save_pretrained(f"{output_dir}/best")
            print(f"  → Saved best (acc={best_acc:.4f})")
        else:
            no_improve += 1
            print(f"  No improve ({no_improve}/{patience})")
            if no_improve >= patience:
                print(f"Early stopping at epoch {epoch+1}")
                break

    print(f"\nBest acc: {best_acc:.4f}")
    return f"{output_dir}/best"

In [ ]:
import random, numpy as np, torch

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False

In [ ]:
seed_results_mse = {}

for seed in [42, 0, 123]:
    set_seed(seed)
    ckpt = train_margin_mse(
        base_checkpoint="/kaggle/input/datasets/tranquanghuy2809/tqhtqh/minilm_stage_a",
        train_path="/kaggle/working/domain_train_with_teacher_scores.jsonl",
        dev_path="/kaggle/input/datasets/tranquanghuy2809/tqhtqh/domain_train_final_dev.jsonl",
        output_dir=f"/kaggle/working/ckpt_mse_seed{seed}",
        epochs=5, batch_size=16, lr=1e-5, patience=2, seed=seed,
    )
    device     = torch.device("cuda")
    tok_eval   = AutoTokenizer.from_pretrained(ckpt)
    model_eval = AutoModelForSequenceClassification.from_pretrained(ckpt).to(device).eval()
    result = benchmark_reranker(
        model=model_eval, tokenizer=tok_eval,
        retrieve_results=retrieve_results, questions=questions,
        device=device, max_questions=len(retrieve_results),
        batch_size=32, max_len=512, label=f"Margin-MSE seed={seed}"
    )
    seed_results_mse[seed] = result
    del model_eval, tok_eval
    torch.cuda.empty_cache()
    print(f"Seed {seed} → Recall@5={result['Recall@5']:.4f} | MRR@10={result['MRR@10']:.4f}")

print(f"\nMRR@10:   {np.mean([r['MRR@10'] for r in seed_results_mse.values()]):.4f} ± {np.std([r['MRR@10'] for r in seed_results_mse.values()]):.4f}")
print(f"Recall@5: {np.mean([r['Recall@5'] for r in seed_results_mse.values()]):.4f} ± {np.std([r['Recall@5'] for r in seed_results_mse.values()]):.4f}")

In [ ]:
# ── Phân tích miss cases chính xác ────────────────────────────────
import json, torch
from collections import Counter
from transformers import AutoTokenizer, AutoModelForSequenceClassification

BEST_CKPT = "/kaggle/working/ckpt_mse_seed42/best"
device    = torch.device("cuda")

tok_eval   = AutoTokenizer.from_pretrained(BEST_CKPT)
model_eval = AutoModelForSequenceClassification.from_pretrained(
    BEST_CKPT
).to(device).eval()

# ── Warmup trước ───────────────────────────────────────────────────
print("Warming up...")
for entry in retrieve_results[:10]:
    query = entry["question"]
    candidates = entry["candidates"]
    pairs = [(query, c["chunk"]) for c in candidates]
    for i in range(0, len(pairs), 32):
        batch = pairs[i:i+32]
        enc = tok_eval(
            [p[0] for p in batch], [p[1] for p in batch],
            max_length=512, padding=True,
            truncation=True, return_tensors="pt"
        ).to(device)
        with torch.no_grad():
            _ = model_eval(**enc).logits
torch.cuda.synchronize()

# ── Collect miss details ───────────────────────────────────────────
miss_details = []

for entry in retrieve_results:
    query = entry["question"]
    if query not in questions:
        continue

    candidates = entry["candidates"]
    gold_ids   = set(questions[query]["gold_chunk_ids"])
    pairs      = [(query, c["chunk"]) for c in candidates]
    scores     = []

    torch.cuda.synchronize()
    for i in range(0, len(pairs), 32):
        batch = pairs[i:i+32]
        enc   = tok_eval(
            [p[0] for p in batch], [p[1] for p in batch],
            max_length=512, padding=True,
            truncation=True, return_tensors="pt"
        ).to(device)
        with torch.no_grad():
            logits = model_eval(**enc).logits.squeeze(-1)
        out = logits.cpu().tolist()
        if isinstance(out, float):
            out = [out]
        scores.extend(out)
    torch.cuda.synchronize()

    ranked = [c for _, c in sorted(zip(scores, candidates), key=lambda x: -x[0])]
    top5   = [c["chunk_id"] for c in ranked[:5]]

    if not any(g in top5 for g in gold_ids):
        gold_rank = None
        for rank, c in enumerate(ranked, 1):
            if c["chunk_id"] in gold_ids:
                gold_rank = rank
                break
        miss_details.append({
            "query":      query,
            "intent":     questions[query].get("intent", ""),
            "gold_ids":   list(gold_ids),
            "top5_ids":   top5,
            "gold_rank":  gold_rank,
            "top1_id":    ranked[0]["chunk_id"],
            "top1_chunk": ranked[0]["chunk"][:200],
        })

del model_eval, tok_eval
torch.cuda.empty_cache()

# ── Stats ──────────────────────────────────────────────────────────
print(f"Total misses: {len(miss_details)}")

rank_dist = Counter(d["gold_rank"] for d in miss_details if d["gold_rank"])
print(f"\nGold rank distribution:")
for r in sorted(rank_dist):
    print(f"  Rank {r:>2}: {rank_dist[r]}")

near_miss  = sum(1 for d in miss_details if d["gold_rank"] and d["gold_rank"] <= 10)
far_miss   = sum(1 for d in miss_details if d["gold_rank"] and d["gold_rank"] > 10)
print(f"\nNear misses (rank 6-10):  {near_miss}")
print(f"Far  misses (rank 11-20): {far_miss}")

# ── Print all 40 miss cases ────────────────────────────────────────
print(f"\n{'='*70}")
print("ALL MISS CASES:")
print(f"{'='*70}")
for i, d in enumerate(miss_details, 1):
    rank_str = f"{d['gold_rank']:>2}" if d['gold_rank'] else "N/A"
    print(f"\n[{i:02d}] Rank {rank_str} | {d['query']}")
    print(f"      Top-1: {d['top1_chunk']}")

In [ ]:
import os

# Check file mMARCO-VI hiện tại
MMARCO = "/kaggle/input/datasets/tranquanghuy2809/tqhtqh/reranker_data/train_triplets.jsonl"

count = 0
with open(MMARCO) as f:
    for line in f:
        count += 1
        if count <= 2:
            import json
            d = json.loads(line)
            print(f"Sample {count}:")
            print(f"  Keys: {list(d.keys())}")
            print(f"  Query: {d.get('query', '')[:80]}")

print(f"\nTotal samples: {count}")

In [ ]:
# ── Score mMARCO-VI với BGE-M3 ─────────────────────────────────────
import json, torch
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForSequenceClassification

BGE_DIR = "/kaggle/input/datasets/tranquanghuy2809/tqhtqh/bge-reranker-v2-m3"
MMARCO  = "/kaggle/input/datasets/tranquanghuy2809/tqhtqh/reranker_data/train_triplets.jsonl"
OUTPUT  = "/kaggle/working/mmarco_with_teacher_scores.jsonl"

device    = torch.device("cuda")
tokenizer = AutoTokenizer.from_pretrained(BGE_DIR)
model     = AutoModelForSequenceClassification.from_pretrained(BGE_DIR).to(device).eval()
print(f"BGE-M3 loaded")

def score_batch(pairs, batch_size=32, max_length=512):
    scores = []
    for i in range(0, len(pairs), batch_size):
        batch = pairs[i:i+batch_size]
        enc   = tokenizer(
            [p[0] for p in batch], [p[1] for p in batch],
            max_length=max_length, padding=True,
            truncation=True, return_tensors="pt"
        ).to(device)
        with torch.no_grad():
            logits = model(**enc).logits.squeeze(-1)
        out = logits.cpu().tolist()
        if isinstance(out, float):
            out = [out]
        scores.extend(out)
    return scores

# Load all data
data = []
with open(MMARCO) as f:
    for line in f:
        data.append(json.loads(line))
print(f"Total: {len(data)}")

# Score positives
print("Scoring positives...")
pos_pairs = [(d["query"], d["positive"]) for d in data]
pos_scores = []
for i in tqdm(range(0, len(pos_pairs), 32)):
    pos_scores.extend(score_batch(pos_pairs[i:i+32], batch_size=32))

# Score negatives
print("Scoring negatives...")
neg_pairs = [(d["query"], d["negative"]) for d in data]
neg_scores = []
for i in tqdm(range(0, len(neg_pairs), 32)):
    neg_scores.extend(score_batch(neg_pairs[i:i+32], batch_size=32))

# Save + compute hardness
import numpy as np

hard = medium = easy = discarded = 0

with open(OUTPUT, "w") as f:
    for d, ps, ns in zip(data, pos_scores, neg_scores):
        margin = ps - ns
        
        if margin < 0.1:
            hardness = "discard"
            discarded += 1
        elif margin < 0.4:
            hardness = "hard"
            hard += 1
        elif margin < 0.7:
            hardness = "medium"
            medium += 1
        else:
            hardness = "easy"
            easy += 1
        
        f.write(json.dumps({
            **d,
            "teacher_pos_score": round(ps, 6),
            "teacher_neg_score": round(ns, 6),
            "teacher_margin":    round(margin, 6),
            "hardness":          hardness,
        }, ensure_ascii=False) + "\n")

margins = np.array(pos_scores) - np.array(neg_scores)
print(f"\nDone! Saved → {OUTPUT}")
print(f"\nHardness distribution:")
print(f"  hard:     {hard:>6} ({hard/len(data)*100:.1f}%)")
print(f"  medium:   {medium:>6} ({medium/len(data)*100:.1f}%)")
print(f"  easy:     {easy:>6} ({easy/len(data)*100:.1f}%)")
print(f"  discard:  {discarded:>6} ({discarded/len(data)*100:.1f}%)")
print(f"\nMargin stats:")
print(f"  Mean: {margins.mean():.4f}")
print(f"  Std:  {margins.std():.4f}")

del model, tokenizer
torch.cuda.empty_cache()

In [ ]:
# ── Mine hard negatives cho mMARCO-VI ─────────────────────────────
import json, torch
import numpy as np
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModel

EMBED_MODEL = "/kaggle/input/datasets/tranquanghuy2809/data-embedding/Vietnamese_Embedding_v2"
MMARCO      = "/kaggle/input/datasets/tranquanghuy2809/tqhtqh/reranker_data/train_triplets.jsonl"
OUTPUT      = "/kaggle/working/mmarco_hard_neg_mined.jsonl"

device    = torch.device("cuda")
tokenizer = AutoTokenizer.from_pretrained(EMBED_MODEL)
embed_model = AutoModel.from_pretrained(EMBED_MODEL).to(device).eval()
print(f"Embedding model loaded")

# ── Mean pooling ───────────────────────────────────────────────────
def mean_pool(model_output, attention_mask):
    token_emb  = model_output.last_hidden_state
    input_mask = attention_mask.unsqueeze(-1).expand(token_emb.size()).float()
    return (token_emb * input_mask).sum(1) / input_mask.sum(1).clamp(min=1e-9)

def embed_batch(texts, batch_size=64, max_length=256):
    all_embs = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        enc   = tokenizer(
            batch, max_length=max_length,
            padding=True, truncation=True,
            return_tensors="pt"
        ).to(device)
        with torch.no_grad():
            out  = embed_model(**enc)
            embs = mean_pool(out, enc["attention_mask"])
            embs = torch.nn.functional.normalize(embs, dim=-1)
        all_embs.append(embs.cpu().numpy())
    return np.vstack(all_embs)

# ── Load mMARCO ────────────────────────────────────────────────────
data = []
with open(MMARCO) as f:
    for line in f:
        data.append(json.loads(line))
print(f"Total: {len(data)}")

# ── Embed tất cả positive chunks ──────────────────────────────────
print("Embedding positive chunks...")
pos_texts = [d["positive"] for d in data]
pos_embs  = embed_batch(pos_texts, batch_size=64)
print(f"Positive embeddings: {pos_embs.shape}")

# ── Mine hard negatives bằng embedding similarity ─────────────────
print("Mining hard negatives...")

# Tính similarity matrix theo batch để tránh OOM
# Với 50k × 50k = 2.5B pairs → cần batch
BATCH = 1000
TOP_K = 10  # lấy top-10 nearest, chọn 3-5 làm neg

results = []

for i in tqdm(range(0, len(data), BATCH)):
    batch_embs = pos_embs[i:i+BATCH]  # (BATCH, dim)
    
    # Cosine similarity với tất cả pos chunks
    sims = batch_embs @ pos_embs.T  # (BATCH, 50k)
    
    for j, sim_row in enumerate(sims):
        idx      = i + j
        query    = data[idx]["query"]
        positive = data[idx]["positive"]
        
        # Lấy top-K nearest chunks (loại chính nó)
        sim_row[idx] = -1  # loại chính nó
        
        # Lấy top-10 nearest positive chunks từ queries khác
        top_indices = np.argsort(sim_row)[::-1][:TOP_K]
        hard_negs   = [data[k]["positive"] for k in top_indices]
        
        # Pair với 3 hard negatives đầu
        for neg in hard_negs[:3]:
            results.append({
                "query":    query,
                "positive": positive,
                "negative": neg,
            })

print(f"Total hard neg pairs: {len(results)}")

# ── Save ───────────────────────────────────────────────────────────
with open(OUTPUT, "w") as f:
    for r in results:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

print(f"Saved → {OUTPUT}")

del embed_model
torch.cuda.empty_cache()

In [ ]:
import json
import numpy as np

OUTPUT = "/kaggle/working/mmarco_hard_neg_mined.jsonl"

# ── Basic stats ────────────────────────────────────────────────────
data = []
with open(OUTPUT) as f:
    for line in f:
        data.append(json.loads(line))

print(f"Total pairs: {len(data)}")

# ── Sample 5 để xem quality ───────────────────────────────────────
import random
random.seed(42)
samples = random.sample(data, 5)

print(f"\n{'='*70}")
print("SAMPLE HARD NEGATIVES:")
print(f"{'='*70}")
for i, d in enumerate(samples, 1):
    print(f"\n[{i}] Query: {d['query'][:80]}")
    print(f"    Positive: {d['positive'][:120]}")
    print(f"    Hard Neg: {d['negative'][:120]}")
    print(f"    Same text? {d['positive'][:50] == d['negative'][:50]}")

# ── Check positive == negative (bug) ──────────────────────────────
same_count = sum(1 for d in data if d["positive"][:100] == d["negative"][:100])
print(f"\nPositive == Negative: {same_count} ({same_count/len(data)*100:.1f}%)")

# ── Check độ dài negative ──────────────────────────────────────────
neg_lens = [len(d["negative"]) for d in data]
print(f"\nNegative length stats:")
print(f"  Mean: {np.mean(neg_lens):.0f} chars")
print(f"  Min:  {min(neg_lens)}")
print(f"  Max:  {max(neg_lens)}")

# ── Unique queries ─────────────────────────────────────────────────
unique_queries = len(set(d["query"] for d in data))
print(f"\nUnique queries: {unique_queries} / {len(data)} pairs")
print(f"Avg pairs/query: {len(data)/unique_queries:.1f}")

In [ ]:
import json
import numpy as np

INPUT  = "/kaggle/working/mmarco_hard_neg_mined.jsonl"
OUTPUT = "/kaggle/working/mmarco_hard_neg_clean.jsonl"

kept = removed_same = removed_short = 0

with open(INPUT) as fin, open(OUTPUT, "w") as fout:
    for line in fin:
        d = json.loads(line)
        
        # Filter 1: bỏ positive == negative
        if d["positive"][:100] == d["negative"][:100]:
            removed_same += 1
            continue
        
        # Filter 2: bỏ negative quá ngắn (< 50 chars)
        if len(d["negative"]) < 50:
            removed_short += 1
            continue
        
        fout.write(line)
        kept += 1

print(f"Kept:           {kept}")
print(f"Removed (same): {removed_same}")
print(f"Removed (short):{removed_short}")
print(f"Total removed:  {removed_same + removed_short}")

In [ ]:
# ── Filter + Score với BGE-M3 ──────────────────────────────────────
import json, torch
import numpy as np
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForSequenceClassification

INPUT    = "/kaggle/working/mmarco_hard_neg_mined.jsonl"
OUTPUT   = "/kaggle/working/mmarco_hard_neg_clean.jsonl"
BGE_DIR  = "/kaggle/input/datasets/tranquanghuy2809/tqhtqh/bge-reranker-v2-m3"

# ── Step 1: Filter ─────────────────────────────────────────────────
kept = removed_same = removed_short = 0
clean_data = []

with open(INPUT) as f:
    for line in f:
        d = json.loads(line)
        if d["positive"][:100] == d["negative"][:100]:
            removed_same += 1
            continue
        if len(d["negative"]) < 50:
            removed_short += 1
            continue
        clean_data.append(d)
        kept += 1

print(f"After filter:")
print(f"  Kept:            {kept}")
print(f"  Removed (same):  {removed_same}")
print(f"  Removed (short): {removed_short}")

# ── Step 2: Score với BGE-M3 ───────────────────────────────────────
device    = torch.device("cuda")
tokenizer = AutoTokenizer.from_pretrained(BGE_DIR)
model     = AutoModelForSequenceClassification.from_pretrained(BGE_DIR).to(device).eval()
print(f"\nBGE-M3 loaded, scoring {len(clean_data)} pairs...")

def score_pairs(queries, docs, batch_size=32, max_length=512):
    scores = []
    pairs  = list(zip(queries, docs))
    for i in range(0, len(pairs), batch_size):
        batch = pairs[i:i+batch_size]
        enc   = tokenizer(
            [p[0] for p in batch], [p[1] for p in batch],
            max_length=max_length, padding=True,
            truncation=True, return_tensors="pt"
        ).to(device)
        with torch.no_grad():
            logits = model(**enc).logits.squeeze(-1)
        out = logits.cpu().tolist()
        if isinstance(out, float):
            out = [out]
        scores.extend(out)
    return scores

# Score positive
print("Scoring positives...")
pos_scores = []
for i in tqdm(range(0, len(clean_data), 32)):
    batch = clean_data[i:i+32]
    pos_scores.extend(score_pairs(
        [d["query"] for d in batch],
        [d["positive"] for d in batch]
    ))

# Score negative
print("Scoring negatives...")
neg_scores = []
for i in tqdm(range(0, len(clean_data), 32)):
    batch = clean_data[i:i+32]
    neg_scores.extend(score_pairs(
        [d["query"] for d in batch],
        [d["negative"] for d in batch]
    ))

# ── Step 3: Compute hardness + Save ───────────────────────────────
margins = np.array(pos_scores) - np.array(neg_scores)

hard = medium = easy = discarded = 0

with open(OUTPUT, "w") as f:
    for d, ps, ns, margin in zip(clean_data, pos_scores, neg_scores, margins):
        if margin < 0.1:
            hardness = "discard"
            discarded += 1
        elif margin < 0.4:
            hardness = "hard"
            hard += 1
        elif margin < 0.7:
            hardness = "medium"
            medium += 1
        else:
            hardness = "easy"
            easy += 1

        f.write(json.dumps({
            **d,
            "teacher_pos_score": round(float(ps), 6),
            "teacher_neg_score": round(float(ns), 6),
            "teacher_margin":    round(float(margin), 6),
            "hardness":          hardness,
        }, ensure_ascii=False) + "\n")

del model, tokenizer
torch.cuda.empty_cache()

# ── Stats ──────────────────────────────────────────────────────────
print(f"\nHardness distribution:")
print(f"  hard:    {hard:>6} ({hard/kept*100:.1f}%)")
print(f"  medium:  {medium:>6} ({medium/kept*100:.1f}%)")
print(f"  easy:    {easy:>6} ({easy/kept*100:.1f}%)")
print(f"  discard: {discarded:>6} ({discarded/kept*100:.1f}%)")

print(f"\nMargin stats:")
print(f"  Mean: {margins.mean():.4f}")
print(f"  Std:  {margins.std():.4f}")
print(f"  Min:  {margins.min():.4f}")
print(f"  Max:  {margins.max():.4f}")

# ── Sample hard cases ──────────────────────────────────────────────
import random
hard_cases = [(d, m) for d, m in zip(clean_data, margins) if 0.1 <= m <= 0.4]
if hard_cases:
    print(f"\nSample hard cases (margin 0.1-0.4):")
    for d, m in random.sample(hard_cases, min(3, len(hard_cases))):
        print(f"\n  Margin: {m:.3f}")
        print(f"  Query:    {d['query'][:80]}")
        print(f"  Positive: {d['positive'][:100]}")
        print(f"  Hard Neg: {d['negative'][:100]}")